In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import os
import re
import pickle
from collections import Counter
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.colors as mcolors
import seaborn as sns
from esda.moran import Moran
from libpysal.weights import DistanceBand
import geopandas as gpd
from scipy.spatial.distance import cdist
from sklearn.preprocessing import MinMaxScaler
import random
from collections import Counter, defaultdict
import shap
from joblib import load
from joblib import Parallel, delayed
from scipy.sparse import csr_matrix
from sklearn.neighbors import BallTree

In [2]:

def minimum_connected_threshold(coordinates):
    """
    Computes the smallest distance such that every point has at least one neighbor.
    """
    tree = BallTree(coordinates, metric='euclidean')
    dists, _ = tree.query(coordinates, k=2)  # k=1 is self, k=2 is nearest neighbor
    nearest_distances = dists[:, 1]  # Take second column (nearest neighbor)
    return np.max(nearest_distances)

def create_sparse_weights_matrix(coordinates, threshold):
    """
    Create a sparse binary spatial weights matrix using a distance threshold.
    """
    distance_matrix = cdist(coordinates, coordinates)
    mask = distance_matrix <= threshold
    np.fill_diagonal(mask, 0)
    return csr_matrix(mask.astype(int))

def create_sparse_weights_balltree(coordinates, threshold):
    tree = BallTree(coordinates, metric='euclidean')
    ind = tree.query_radius(coordinates, r=threshold)
    row_idx = []
    col_idx = []

    for i, neighbors in enumerate(ind):
        for j in neighbors:
            if i != j:
                row_idx.append(i)
                col_idx.append(j)

    data = np.ones(len(row_idx), dtype=int)
    return csr_matrix((data, (row_idx, col_idx)), shape=(len(coordinates), len(coordinates)))

def morans_I_vectorized(values, spatial_weights):
    N = len(values)
    W = spatial_weights.sum()
    mean_value = np.mean(values)
    diff = values - mean_value

    # Sparse matrix multiplication
    numerator = diff @ spatial_weights.dot(diff)
    denominator = np.sum(diff ** 2)

    return (N / W) * (numerator / denominator)

def calculate_p_value(residuals, spatial_weights, observed_I, permutations=999):
    N = len(residuals)
    mean_value = np.mean(residuals)
    diff = residuals - mean_value
    denominator = np.sum(diff ** 2)

    W = spatial_weights.sum()

    random_I_values = np.zeros(permutations)
    for i in range(permutations):
        permuted = np.random.permutation(diff)
        numerator = permuted @ spatial_weights.dot(permuted)
        random_I_values[i] = (N / W) * (numerator / denominator)

    p_value = (np.sum(np.abs(random_I_values) >= np.abs(observed_I)) + 1) / (permutations + 1)
    return p_value

def morans_I(df, residuals, threshold=None, crs_epsg=3395, permutations=999):
    """
    Calculate Moran's I with a p-value using a minimum connected threshold if not given.
    """
    # Step 1: Convert to GeoDataFrame and reproject
    gdf = gpd.GeoDataFrame(
        df,
        geometry=gpd.points_from_xy(df['Longitude_0'], df['Latitude_0']),
        crs="EPSG:4326"
    )
    gdf = gdf.to_crs(epsg=crs_epsg)
    coordinates = np.array([(point.x, point.y) for point in gdf.geometry])

    # Step 2: Use minimum connected threshold if not provided
    if threshold is None:
        threshold = minimum_connected_threshold(coordinates)
        print(f"[INFO] Minimum connected threshold computed: {threshold:.2f} meters")

    # Step 3: Create spatial weights using BallTree method
    weights_matrix = create_sparse_weights_balltree(coordinates, threshold)

    # Step 4: Compute Moran's I and p-value
    observed_I = morans_I_vectorized(residuals, weights_matrix)
    p_value = calculate_p_value(residuals, weights_matrix, observed_I, permutations)

    return observed_I, p_value

In [3]:
PASTEL_version = "v0_1_5"

In [4]:
df = pd.read_csv(r"C:\Users\vwgei\Documents\PVOCAL\data\v0_1_5\df_preprocessed_backup.csv")

In [5]:
df['Rainfall_Estimate_0'].describe()

count    40040.0
mean         0.0
std          0.0
min          0.0
25%          0.0
50%          0.0
75%          0.0
max          0.0
Name: Rainfall_Estimate_0, dtype: float64

In [6]:
# Convert -180 to 180 to 0-360
df['Longitude_0'] = df['Longitude_0'].apply(lambda x: (x + 360) if x < 0 else x)

In [ ]:
# Shift longitude values so that -180 and 180 are treated as neighbors
df['Longitude_shifted'] = df['Longitude_0'].apply(lambda x: x % 360)

In [ ]:
df['Datetime'] = pd.to_datetime(df['Datetime'], format='%m/%d/%Y %H:%M')

In [ ]:
CO = df['CO']

In [ ]:
df = df.sort_values(by='Datetime')

In [ ]:
df['time_index'] = range(len(df))

In [ ]:
for c in df.columns:
    print(c)

In [ ]:
df['sum_moisture_flux'].max()

In [ ]:
# use in title longname\ninputfeaturename
input_features = [
    'mean_distance_to_t1000_cities', 'min_distance_to_t1000_cities',
    'Latitude_0', 'Longitude_0', 'AltP_meters_0',
    'Specific_Humidity_0', 'Potential_Temperature_0', 'mean_lcl', 'bearings_from_origin_-24',
    'sum_moisture_flux', 'mean_mixing_depth', 'sum_solar_radiation', 'domain_indicator', 
    'time_difference_seconds_0', 'cos_julian_day', 'hour_cos_0', 
]

# use in colorbar
input_feature_units = [
    'kilometers', 'kilometers', '° (degrees)', '° (degrees)', 'meters', 'g/kg',
    'K (Kelvin)', 'meters', 'radians', '(g * m)/(kg * s)', 'meters', 'watts/(meter^2)',
    'unitless', 's (seconds)', 'unitless', 'unitless'
]

# use in title
input_feature_long_name = [
    'Average Distance to Top 1000 Most Populated Cites', 'Minimum Distance to Top 1000 Most Populated Cites',
    'Degree Latitude', 'Degree Longitude', 'Altitude', 'Specific Humidity', 'Potential Temperature',
    'Mean Lifted Condensation Level (LCL)', '-24 Hour Bearing From Trajectory Origin', 'Total Trajectory Moisture Flux',
    'Average Mixing Depth', 'Total Solar Radiation', 'Domain Indicator', 'Seconds Elapsed Since 01/01/1900 00:00:00',
    'cosine(Julian Day)', 'cosine(Hour of Day)'
]

In [ ]:
os.chdir(r"C:\Users\vwgei\Documents\PVOCAL\plots\standalone\input_vars")

In [ ]:
# Loop through each feature and generate a plot
for i, var in enumerate(input_features):
    # Create the plot
    fig, ax = plt.subplots(figsize=(12, 8), subplot_kw={'projection': ccrs.PlateCarree(central_longitude=240)})
    ax.set_extent([-136, 120, -90, 90], crs=ccrs.PlateCarree(central_longitude=240))

    # Add basemap features
    ax.coastlines(resolution='50m')
    ax.add_feature(cfeature.BORDERS.with_scale('50m'), linestyle=':')
    ax.add_feature(cfeature.LAND.with_scale('50m'), facecolor='lightgray')
    ax.add_feature(cfeature.OCEAN.with_scale('50m'), facecolor='white')

    # Add gridlines
    gl = ax.gridlines(draw_labels=True, linestyle="--", color="gray")
    gl.top_labels = gl.right_labels = False  # Optional: turn off top/right labels for cleaner look

    # Plot the data with scatter plot
    sc = ax.scatter(
        df['Longitude_0'].values, 
        df['Latitude_0'].values, 
        c=df[var].values,  # Use numeric values for consistent coloring
        cmap='jet', 
        s=20, 
        transform=ccrs.PlateCarree(),
    )

    # Add the colorbar
    cbar = plt.colorbar(sc, ax=ax, orientation='vertical', aspect=30, shrink=0.7)
    cbar.set_label(f'{input_feature_units[i]}')

    # Title and labels
    ax.set_title(f"{input_feature_long_name[i]}\n({input_features[i]})")

    # Save the plot for each feature
    # Clean the variable name to remove invalid characters
    safe_var = re.sub(r'[<>:"/\\|?*]', '_', str(var))  # Replace invalid characters with '_'

    plt.savefig(rf"C:\Users\vwgei\Documents\PVOCAL\plots\standalone\input_vars\{safe_var}.png", dpi=300, bbox_inches='tight')
    # plt.show()
    plt.clf()
    plt.close()

In [7]:
for c in df.columns:
    print(c)

iindex
LAT
LON
ALT
Datetime
Datetime_old
CH4
CO2
CO
O3
Ethane
DMS
CH3Br
Year
Month
Day
Hour
domain_indicator
Pressure_-24
Potential_Temperature_-24
Temperature_-24
Rainfall_-24
Mixing_Depth_-24
Relative_Humidity_-24
Specific_Humidity_-24
Mixing_Ratio_-24
Terrain_Altitude_-24
Solar_Radiation_-24
geometry_-24
DateTime_-24
Temperature_C_-24
Distance_ptp_-24
Cumulative_Dist_-24
Dist_from_origin_-24
bearings_from_origin_-24
bearings_ptp_-24
Moisture_Flux_-24
Pressure_-23
Potential_Temperature_-23
Temperature_-23
Rainfall_-23
Mixing_Depth_-23
Relative_Humidity_-23
Specific_Humidity_-23
Mixing_Ratio_-23
Terrain_Altitude_-23
Solar_Radiation_-23
geometry_-23
DateTime_-23
Temperature_C_-23
Distance_ptp_-23
Cumulative_Dist_-23
Dist_from_origin_-23
bearings_from_origin_-23
bearings_ptp_-23
Moisture_Flux_-23
Pressure_-22
Potential_Temperature_-22
Temperature_-22
Rainfall_-22
Mixing_Depth_-22
Relative_Humidity_-22
Specific_Humidity_-22
Mixing_Ratio_-22
Terrain_Altitude_-22
Solar_Radiation_-22
geomet

In [11]:
target_features = [
    'DMS', 'CO', 'CH4', 'CH3Br', 'O3', 'Ethane'
]

# use in colorbar
target_feature_units = [
    'pptv', 'ppbv', 'ppbv', 'pptv', 'ppbv', 'pptv'
]

# use in title
target_feature_long_name = [
    'Dimethyl Sulfide', 'Carbon Monoxide', 'Methane', 'Methyl Bromide', 'Ozone', 'Ethane'
]

# Loop through each feature and generate a plot
for i, var in enumerate(target_features):
    fig, ax = plt.subplots(
        figsize=(12, 8),
        subplot_kw={'projection': ccrs.PlateCarree(central_longitude=240)}
    )
    ax.set_extent([-136, 120, -90, 90], crs=ccrs.PlateCarree(central_longitude=240))

    # Basemap
    ax.coastlines(resolution='50m')
    ax.add_feature(cfeature.BORDERS.with_scale('50m'), linestyle=':')
    ax.add_feature(cfeature.LAND.with_scale('50m'), facecolor='lightgray')
    ax.add_feature(cfeature.OCEAN.with_scale('50m'), facecolor='white')

    # Gridlines
    gl = ax.gridlines(draw_labels=True, linestyle="--", color="gray")
    gl.top_labels = gl.right_labels = False

    # Plot with log normalization
    sc = ax.scatter(
        df['Longitude_0'].values,
        df['Latitude_0'].values,
        c=df[var].values,
        cmap='viridis',
        s=20,
        transform=ccrs.PlateCarree(),
        norm=mcolors.LogNorm(vmin=df[var].min(), vmax=df[var].max())  # log scale
    )

    # Colorbar with log ticks
    cbar = plt.colorbar(sc, ax=ax, orientation='vertical', aspect=30, shrink=0.7)
    cbar.set_label(f'{target_feature_units[i]}')

    # Add both major and minor ticks
    cbar.locator = mticker.LogLocator(base=10, subs=np.arange(1, 10)*0.1, numticks=10)
    cbar.update_ticks()

    # Title
    ax.set_title(f"ATS: {target_feature_long_name[i]}")

    # Save
    safe_var = re.sub(r'[<>:\"/\\|?*]', '_', str(var))
    plt.savefig(rf"C:\Users\vwgei\Documents\PVOCAL\plots\standalone\{safe_var}.png",
                dpi=300, bbox_inches='tight')
    plt.clf()
    plt.close()

In [ ]:
# Open the file in binary read mode and load the object
with open(rf"C:\Users\vwgei\Documents\PVOCAL\ensemble\global_models\{PASTEL_version}_DMS\data\DMS_global_X_train.pkl", 'rb') as file:
    dms_train = pickle.load(file)
with open(rf"C:\Users\vwgei\Documents\PVOCAL\ensemble\global_models\{PASTEL_version}_DMS\data\DMS_global_X_test.pkl", 'rb') as file:
    dms_test = pickle.load(file)

with open(rf"C:\Users\vwgei\Documents\PVOCAL\ensemble\global_models\{PASTEL_version}_DMS\data\DMS_global_X_train_iindex.pkl", 'rb') as file:
    dms_train_iindex = pickle.load(file)
with open(rf"C:\Users\vwgei\Documents\PVOCAL\ensemble\global_models\{PASTEL_version}_DMS\data\DMS_global_X_test_iindex.pkl", 'rb') as file:
    dms_test_iindex = pickle.load(file) 

dms_residual_train = np.load(rf"C:\Users\vwgei\Documents\PVOCAL\ensemble\PASTEL_combined\{PASTEL_version}_DMS\DMS_PASTEL_combined_residuals_train.npy")
dms_residual_test = np.load(rf"C:\Users\vwgei\Documents\PVOCAL\ensemble\PASTEL_combined\{PASTEL_version}_DMS\DMS_PASTEL_combined_residuals_test.npy")
dms_pred_train = np.load(rf"C:\Users\vwgei\Documents\PVOCAL\ensemble\PASTEL_combined\{PASTEL_version}_DMS\DMS_PASTEL_combined_pred_train.npy")
dms_pred_test = np.load(rf"C:\Users\vwgei\Documents\PVOCAL\ensemble\PASTEL_combined\{PASTEL_version}_DMS\DMS_PASTEL_combined_pred_test.npy")

# dms_train['iindex'] = dms_train_iindex
# dms_test['iindex'] = dms_test_iindex

# merged_dms = dms_test.merge(df[['Latitude_0', 'Longitude_0']], left_on='iindex', right_index=True, how='left')


# Combine the residuals from both datasets to compute vmin and vmax
all_residuals = dms_residual_test #np.concatenate([dms_residual_train, dms_residual_test]) 

# Calculate the required statistics
mean_residual = np.mean(all_residuals)
median_residual = np.median(all_residuals)
std_residual = np.std(all_residuals)
above_2sd = np.sum(all_residuals > mean_residual + 2 * std_residual)
below_2sd = np.sum(all_residuals < mean_residual - 2 * std_residual)
n = len(all_residuals)

# Set up the color normalization (using linear scale)
norm = mcolors.Normalize(vmin=mean_residual - 2 * std_residual, vmax=mean_residual + 2 * std_residual)

# Create the plot
fig, ax = plt.subplots(figsize=(12, 8), subplot_kw={'projection': ccrs.PlateCarree(central_longitude=240)})
ax.set_extent([-136, 120, -90, 90], crs=ccrs.PlateCarree(central_longitude=240))

# Add basemap features
ax.coastlines(resolution='50m')
ax.add_feature(cfeature.BORDERS.with_scale('50m'), linestyle=':')
ax.add_feature(cfeature.LAND.with_scale('50m'), facecolor='white')
ax.add_feature(cfeature.OCEAN.with_scale('50m'), facecolor='white')

# Add gridlines
gl = ax.gridlines(draw_labels=True, linestyle="--", color="gray")
gl.top_labels = gl.right_labels = False  # optional: turn off top/right labels for cleaner look

# # Plot the data for the training set
# sc0 = ax.scatter(
#     dms_train['Longitude_0'].values, 
#     dms_train['Latitude_0'].values, 
#     c=dms_residual_train,  # Use numeric residuals for consistent coloring
#     cmap='coolwarm_r', 
#     s=20, 
#     transform=ccrs.PlateCarree(),
#     norm=norm  # Apply the same normalization
# )

# Plot the data for the test set
sc1 = ax.scatter(
    dms_test['Longitude_0'].values, 
    dms_test['Latitude_0'].values, 
    c=dms_residual_test,  # Use numeric residuals for consistent coloring
    cmap='coolwarm_r', 
    s=20, 
    transform=ccrs.PlateCarree(),
    norm=norm  # Apply the same normalization
)

# Add the shared colorbar
cbar = plt.colorbar(sc1, ax=ax, orientation='vertical', pad=0.05, aspect=30, shrink=0.7, extend='both')
cbar.set_label(f'DMS (pptv)')

# Title and labels
ax.set_title(f"PASTEL Residuals\n[Test set]")

# Annotate the statistics in the lower left corner
stats_text = (f"n: {n:.0f}\n"
              f"Median: {median_residual:.2f}\n"
              f"Mean: {mean_residual:.2f}\n"
              f"Std Dev: {std_residual:.2f}\n"
              f"# > 2 Std Dev: {above_2sd}\n"
              f"# < -2 Std Dev: {below_2sd}")
ax.text(-130, -85, stats_text, fontsize=12, color='black', ha='left', va='bottom', backgroundcolor='white')

# Save and display the plot
plt.savefig(rf"C:\Users\vwgei\Documents\PVOCAL\plots\DMS\DMS_residuals_test.png", dpi=300, bbox_inches='tight')
plt.tight_layout()
plt.show()

# Combine the residuals from both datasets to compute vmin and vmax
all_pred = dms_pred_test#np.concatenate([dms_pred_train, dms_pred_test])

# Calculate the required statistics
mean_pred = np.mean(all_pred)
median_pred = np.median(all_pred)
std_pred = np.std(all_pred)
above_2sd = np.sum(all_pred > mean_pred + 2 * std_pred)
below_2sd = np.sum(all_pred < mean_pred - 2 * std_pred)
n = len(all_pred)

min = np.min(all_pred)
max = np.max(all_pred)

# Set up the color normalization (using linear scale)
norm = mcolors.LogNorm(vmin=1, vmax=dms_pred_test.max())

# Create the plot
fig, ax = plt.subplots(figsize=(12, 8), subplot_kw={'projection': ccrs.PlateCarree(central_longitude=240)})
ax.set_extent([-136, 120, -90, 90], crs=ccrs.PlateCarree(central_longitude=240))

# Add basemap features
ax.coastlines(resolution='50m')
ax.add_feature(cfeature.BORDERS.with_scale('50m'), linestyle=':')
ax.add_feature(cfeature.LAND.with_scale('50m'), facecolor='white')
ax.add_feature(cfeature.OCEAN.with_scale('50m'), facecolor='white')

# Add gridlines
gl = ax.gridlines(draw_labels=True, linestyle="--", color="gray")
gl.top_labels = gl.right_labels = False  # optional: turn off top/right labels for cleaner look

# # Plot the data for the training set
# sc0 = ax.scatter(
#     dms_train['Longitude_0'].values, 
#     dms_train['Latitude_0'].values, 
#     c=dms_pred_train,  # Use numeric residuals for consistent coloring
#     cmap='viridis', 
#     s=20, 
#     transform=ccrs.PlateCarree(),
#     norm=norm  # Apply the same normalization
# )

# Plot the ata for the test set
sc1 = ax.scatter(
    dms_test['Longitude_0'].values, 
    dms_test['Latitude_0'].values, 
    c=dms_pred_test,  # Use numeric residuals for consistent coloring
    cmap='viridis', 
    s=20, 
    transform=ccrs.PlateCarree(),
    norm=norm  # Apply the same normalization
)

# Add the shared colorbar
cbar = plt.colorbar(sc1, ax=ax, orientation='vertical', pad=0.05, aspect=30, shrink=0.7, extend='min')
cbar.set_label(f'DMS (pptv)')

# Title and labels
ax.set_title(f"PASTEL Predicted DMS\n[Test Set]")

# Annotate the statistics in the lower left corner
stats_text = (f"n: {n:.0f}\n"
              f"Median: {median_pred:.2f}\n"
              f"Mean: {mean_pred:.2f}\n"
              f"Std Dev: {std_pred:.2f}\n"
              f"# > 2 Std Dev: {above_2sd}\n"
            #   f"< -2 Std Dev: {below_2sd}"
            )
ax.text(-130, -85, stats_text, fontsize=12, color='black', ha='left', va='bottom', backgroundcolor='white')

# Save and display the plot
plt.savefig(rf"C:\Users\vwgei\Documents\PVOCAL\plots\DMS\DMS_combined_pred_test.png", dpi=300, bbox_inches='tight')
plt.tight_layout()
plt.show()

In [ ]:
dms_snr = np.var(dms_pred_test) / np.var(dms_residual_test)
dms_snr

In [ ]:
from sklearn.feature_selection import mutual_info_regression
dms_test['residuals'] = dms_residual_test
dms_test.dropna(inplace=True)
# Compute mutual information between residuals and other features
mutual_info = mutual_info_regression(dms_test.drop('residuals', axis=1), dms_test['residuals'])

# Create a DataFrame to pair features with their mutual information scores
mutual_info_df = pd.DataFrame({
    'Feature': dms_test.drop('residuals', axis=1).columns,  # Feature names
    'Mutual Information': mutual_info  # Corresponding mutual information scores
})

# Sort the DataFrame by mutual information scores in descending order
mutual_info_df = mutual_info_df.sort_values(by='Mutual Information', ascending=False)

# mutual_info_df.drop(15, inplace=True)

# Plot the mutual information scores
plt.figure(figsize=(10, 6))
sns.barplot(x='Mutual Information', y='Feature', data=mutual_info_df, palette='coolwarm', order=mutual_info_df['Feature'])
plt.title('Mutual Information between Features and Residuals')
plt.xlabel('Mutual Information')
plt.ylabel('Feature')
plt.show()
print(mutual_info_df)

with open(os.path.join(rf"C:\Users\vwgei\Documents\PVOCAL\ensemble\PASTEL_combined\{PASTEL_version}_DMS", f"combined_MI_residuals.pkl"), "wb") as f:
    pickle.dump(mutual_info_df, f)

# Calculate the correlation matrix for all features with respect to residuals
correlation_matrix = dms_test.corr(method='pearson')

# Extract the correlation values for 'residuals' with all other variables
residuals_corr = correlation_matrix['residuals']

# Sort the correlations to find the most positively or negatively correlated variables
sorted_residuals_corr = residuals_corr.sort_values(ascending=False)

# Print the sorted correlations
print(sorted_residuals_corr)

# Create a heatmap of the correlation matrix (for visualization)
plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title("Correlation Heatmap")
plt.show()

In [ ]:
# ats = ['CO', 'Ethane', 'O3', 'CH4', 'DMS', 'CH3Br']
# for a in ats:
#     combined_residuals = np.load(rf"C:\Users\vwgei\Documents\PVOCAL\ensemble\PASTEL_combined\v0_1_5_{a}\{a}_PASTEL_combined_residuals_test.npy")
#     combined_pred_test = np.load(rf"C:\Users\vwgei\Documents\PVOCAL\ensemble\PASTEL_combined\v0_1_5_{a}\{a}_PASTEL_combined_pred_test.npy")

#     with open(rf"C:\Users\vwgei\Documents\PVOCAL\ensemble\regional_ensemble\v0_1_5_{a}\{a}_re_residuals_test.pkl", 'rb') as file:
#         regional_residuals = pickle.load(file)
#     regional_pred_test = np.load(rf"C:\Users\vwgei\Documents\PVOCAL\ensemble\regional_ensemble\v0_1_5_{a}\{a}_re_test_predictions_native.npy")

#     with open(rf"C:\Users\vwgei\Documents\PVOCAL\ensemble\global_models\v0_1_5_{a}\data\{a}_global_test_residuals.pkl", 'rb') as file:
#         global_residuals = pickle.load(file)
#     global_pred_test = np.load(rf"C:\Users\vwgei\Documents\PVOCAL\ensemble\global_models\v0_1_5_{a}\data\{a}_global_y_pred_test.npy")
#     # Plot residuals vs. fitted values
#     plt.figure(figsize=(8, 6))
#     plt.scatter(x=global_pred_test, y=global_residuals, c="blue", label="Global Model", marker=".", alpha=0.5)
#     plt.scatter(x=regional_pred_test, y=regional_residuals, c="green", label="Regional Ensemble", marker=".", alpha=0.5)
#     plt.scatter(x=combined_pred_test, y=combined_residuals, c="black", label="Combined Model", marker=".", alpha=0.5)
#     plt.xlabel('Fitted Values')
#     plt.ylabel('Residuals')
#     plt.title(f'Residuals vs. Fitted Values ({a})')
#     plt.show()
#     plt.clf()
#     plt.close()

In [ ]:
ats = "DMS"

# Open the file in binary read mode and load the object
with open(rf"C:\Users\vwgei\Documents\PVOCAL\ensemble\{ats}_cluster_stats.pkl", 'rb') as file:
    var_cluster_stats = pickle.load(file)

test_mi = []
for i in var_cluster_stats.keys():
    test_mi.append(var_cluster_stats[i]['moran_I_test_RF'])

print(f"Average RF testing Morans I: {np.nanmean(test_mi)}")

input_features = [
    'mean_distance_to_t1000_cities', 'min_distance_to_t1000_cities',
    'Latitude_0', 'Longitude_0', 'AltP_meters_0',
    'Specific_Humidity_0', 'Potential_Temperature_0', 'mean_lcl', 'bearings_from_origin_-24',
    'sum_moisture_flux', 'mean_mixing_depth', 'sum_solar_radiation', 'domain_indicator', 
    'time_difference_seconds_0', 'cos_julian_day', 'hour_cos_0', 
]

var_cluster_stats[1]['sorted_idx']

efid = []
for i in var_cluster_stats.keys():
    efid.append(var_cluster_stats[i]['sorted_idx'])

# Extract the top 3 feature indices across all ensemble members
top_features = []
for entry in efid:
    top_features.extend(entry[-1:])  # Get the last 3 indices

# Convert indices to feature names
top_feature_names = [input_features[i] for i in top_features]

# Count occurrences of each feature
feature_counts = Counter(top_feature_names)

# Sort the feature counts by frequency (value) in descending order
sorted_features = sorted(feature_counts.items(), key=lambda x: x[1], reverse=True)
sorted_feature_names, sorted_counts = zip(*sorted_features)

# Plot sorted histogram as horizontal bars
plt.figure(figsize=(10, 6))
bars = plt.barh(sorted_feature_names, sorted_counts, color='purple')

# Add value labels on bars
for bar, count in zip(bars, sorted_counts):
    plt.text(
        bar.get_width() + 0.1,  # X-coordinate of text (just beyond the bar's length)
        bar.get_y() + bar.get_height() / 2,  # Y-coordinate of text (centered vertically on the bar)
        str(count),  # Text to display
        va='center', color='gray', fontsize=10
    )

plt.gca().invert_yaxis()  # Invert Y-axis to have the most important feature at the top
plt.title(f'Histogram of Top Feature Across {ats} Regional Ensemble')
plt.xlabel('Frequency')
plt.ylabel('Input Features')
plt.savefig(rf"C:\Users\vwgei\Documents\PVOCAL\ensemble\regional_ensemble\{PASTEL_version}_{ats}\{ats}_re_feature_importances_composite.png", dpi=300, bbox_inches='tight')
plt.tight_layout()
plt.show()

In [ ]:
# Open the file in binary read mode and load the object
with open(rf"C:\Users\vwgei\Documents\PVOCAL\ensemble\global_models\{PASTEL_version}_Ethane\data\Ethane_global_X_train.pkl", 'rb') as file:
    ethane_train = pickle.load(file)
with open(rf"C:\Users\vwgei\Documents\PVOCAL\ensemble\global_models\{PASTEL_version}_Ethane\data\Ethane_global_X_test.pkl", 'rb') as file:
    ethane_test = pickle.load(file)

with open(rf"C:\Users\vwgei\Documents\PVOCAL\ensemble\global_models\{PASTEL_version}_Ethane\data\Ethane_global_X_train_iindex.pkl", 'rb') as file:
    ethane_train_iindex = pickle.load(file)
with open(rf"C:\Users\vwgei\Documents\PVOCAL\ensemble\global_models\{PASTEL_version}_Ethane\data\Ethane_global_X_test_iindex.pkl", 'rb') as file:
    ethane_test_iindex = pickle.load(file) 

ethane_residual_train = np.load(rf"C:\Users\vwgei\Documents\PVOCAL\ensemble\PASTEL_combined\{PASTEL_version}_Ethane\Ethane_PASTEL_combined_residuals_train.npy")
ethane_residual_test = np.load(rf"C:\Users\vwgei\Documents\PVOCAL\ensemble\PASTEL_combined\{PASTEL_version}_Ethane\Ethane_PASTEL_combined_residuals_test.npy")
ethane_pred_train = np.load(rf"C:\Users\vwgei\Documents\PVOCAL\ensemble\PASTEL_combined\{PASTEL_version}_Ethane\Ethane_PASTEL_combined_pred_train.npy")
ethane_pred_test = np.load(rf"C:\Users\vwgei\Documents\PVOCAL\ensemble\PASTEL_combined\{PASTEL_version}_Ethane\Ethane_PASTEL_combined_pred_test.npy")

# ethane_test['iindex'] = ethane_test.index

# merged_ethane = ethane_test.merge(df[['Latitude_0', 'Longitude_0']], left_on='iindex', right_index=True, how='left')

# Combine the residuals from both datasets to compute vmin and vmax
all_residuals = ethane_residual_test #np.concatenate([dms_residual_train, dms_residual_test])

# Calculate the required statistics
mean_residual = np.mean(all_residuals)
median_residual = np.median(all_residuals)
std_residual = np.std(all_residuals)
above_2sd = np.sum(all_residuals > mean_residual + 2 * std_residual)
below_2sd = np.sum(all_residuals < mean_residual - 2 * std_residual)
n = len(all_residuals)

# Set up the color normalization (using linear scale)
norm = mcolors.Normalize(vmin=mean_residual - 2 * std_residual, vmax=mean_residual + 2 * std_residual)

# Create the plot
fig, ax = plt.subplots(figsize=(12, 8), subplot_kw={'projection': ccrs.PlateCarree(central_longitude=240)})
ax.set_extent([-136, 120, -90, 90], crs=ccrs.PlateCarree(central_longitude=240))

# Add basemap features
ax.coastlines(resolution='50m')
ax.add_feature(cfeature.BORDERS.with_scale('50m'), linestyle=':')
ax.add_feature(cfeature.LAND.with_scale('50m'), facecolor='white')
ax.add_feature(cfeature.OCEAN.with_scale('50m'), facecolor='white')

# Add gridlines
gl = ax.gridlines(draw_labels=True, linestyle="--", color="gray")
gl.top_labels = gl.right_labels = False  # optional: turn off top/right labels for cleaner look

# # Plot the data for the training set
# sc0 = ax.scatter(
#     dms_train['Longitude_0'].values, 
#     dms_train['Latitude_0'].values, 
#     c=dms_residual_train,  # Use numeric residuals for consistent coloring
#     cmap='coolwarm_r', 
#     s=20, 
#     transform=ccrs.PlateCarree(),
#     norm=norm  # Apply the same normalization
# )

# Plot the data for the test set
sc1 = ax.scatter(
    ethane_test['Longitude_0'].values, 
    ethane_test['Latitude_0'].values, 
    c=ethane_residual_test,  # Use numeric residuals for consistent coloring
    cmap='coolwarm_r', 
    s=20, 
    transform=ccrs.PlateCarree(),
    norm=norm  # Apply the same normalization
)

# Add the shared colorbar
cbar = plt.colorbar(sc1, ax=ax, orientation='vertical', pad=0.05, aspect=30, shrink=0.7, extend='both')
cbar.set_label(f'Ethane (pptv)')

# Title and labels
ax.set_title(f"PASTEL Residuals\n[Test set]")

# Annotate the statistics in the lower left corner
stats_text = (f"n: {n:.0f}\n"
              f"Median: {median_residual:.2f}\n"
              f"Mean: {mean_residual:.2f}\n"
              f"Std Dev: {std_residual:.2f}\n"
              f"# > 2 Std Dev: {above_2sd}\n"
              f"# < -2 Std Dev: {below_2sd}")
ax.text(-130, -85, stats_text, fontsize=12, color='black', ha='left', va='bottom', backgroundcolor='white')

# Save and display the plot
plt.savefig(rf"C:\Users\vwgei\Documents\PVOCAL\plots\Ethane\Ethane_residuals_test.png", dpi=300, bbox_inches='tight')
plt.tight_layout()
plt.show()

# Combine the residuals from both datasets to compute vmin and vmax
all_pred = ethane_pred_test#np.concatenate([dms_pred_train, dms_pred_test])

# Calculate the required statistics
mean_pred = np.mean(all_pred)
median_pred = np.median(all_pred)
std_pred = np.std(all_pred)
above_2sd = np.sum(all_pred > mean_pred + 2 * std_pred)
below_2sd = np.sum(all_pred < mean_pred - 2 * std_pred)
n = len(all_pred)

min = np.min(all_pred)
max = np.max(all_pred)


norm = mcolors.LogNorm(vmin=60, vmax=np.max(ethane_pred_test)) #np.min(ethane_pred_test)

# Create the plot
fig, ax = plt.subplots(figsize=(12, 8), subplot_kw={'projection': ccrs.PlateCarree(central_longitude=240)})
ax.set_extent([-136, 120, -90, 90], crs=ccrs.PlateCarree(central_longitude=240))

# Add basemap features
ax.coastlines(resolution='50m')
ax.add_feature(cfeature.BORDERS.with_scale('50m'), linestyle=':')
ax.add_feature(cfeature.LAND.with_scale('50m'), facecolor='white')
ax.add_feature(cfeature.OCEAN.with_scale('50m'), facecolor='white')

# Add gridlines
gl = ax.gridlines(draw_labels=True, linestyle="--", color="gray")
gl.top_labels = gl.right_labels = False  # optional: turn off top/right labels for cleaner look

# # Plot the data for the training set
# sc0 = ax.scatter(
#     dms_train['Longitude_0'].values, 
#     dms_train['Latitude_0'].values, 
#     c=dms_pred_train,  # Use numeric residuals for consistent coloring
#     cmap='viridis', 
#     s=20, 
#     transform=ccrs.PlateCarree(),
#     norm=norm  # Apply the same normalization
# )

# Plot the data for the test set
sc1 = ax.scatter(
    ethane_test['Longitude_0'].values, 
    ethane_test['Latitude_0'].values, 
    c=ethane_pred_test,  # Use numeric residuals for consistent coloring
    cmap='viridis', 
    s=20, 
    transform=ccrs.PlateCarree(),
    norm=norm  # Apply the same normalization
)

# Add the shared colorbar
cbar = plt.colorbar(sc1, ax=ax, orientation='vertical', pad=0.05, aspect=30, shrink=0.7,) #extend='both')
cbar.set_label(f'Ethane (pptv)')

# Title and labels
ax.set_title(f"PASTEL Predicted Ethane\n[Test Set]")

# Annotate the statistics in the lower left corner
stats_text = (f"n: {n:.0f}\n"
              f"Median: {median_pred:.2f}\n"
              f"Mean: {mean_pred:.2f}\n"
              f"Std Dev: {std_pred:.2f}\n"
              f"# > 2 Std Dev: {above_2sd}\n"
            #   f"< -2 Std Dev: {below_2sd}"
            )
ax.text(-130, -85, stats_text, fontsize=12, color='black', ha='left', va='bottom', backgroundcolor='white')

# Save and display the plot
plt.savefig(rf"C:\Users\vwgei\Documents\PVOCAL\plots\Ethane\Ethane_combined_pred_test.png", dpi=300, bbox_inches='tight')
plt.tight_layout()
plt.show()

In [ ]:
ethane_snr = np.var(ethane_pred_test) / np.var(ethane_residual_test)
ethane_snr

In [ ]:
print(len(ethane_train))
print(len(ethane_test))

In [ ]:
ethane_test['pred'] = ethane_pred_test

In [ ]:
# Assuming ethane_test is your dataset
# Split the dataset into two hemispheres
north_hemisphere = ethane_test[ethane_test['Latitude_0'] > 0]
south_hemisphere = ethane_test[ethane_test['Latitude_0'] < 0]

# Calculate the average residuals for each hemisphere
average_modeled_north = north_hemisphere['pred'].mean()
average_modeled_south = south_hemisphere['pred'].mean()

# Output the results
print(f"Average ethane in the Northern Hemisphere: {average_modeled_north}")
print(f"Average ethane in the Southern Hemisphere: {average_modeled_south}")

In [ ]:
north_hemisphere = df[df['Latitude_0'] > 0]
south_hemisphere = df[df['Latitude_0'] < 0]

# Calculate the average residuals for each hemisphere
average_observed_north = north_hemisphere['Ethane'].mean()
average_observed_south = south_hemisphere['Ethane'].mean()

# Output the results
print(f"Average ethane in the Northern Hemisphere: {average_observed_north}")
print(f"Average ethane in the Southern Hemisphere: {average_observed_south}")

In [ ]:
from sklearn.feature_selection import mutual_info_regression
ethane_test['residuals'] = ethane_residual_test
ethane_test.dropna(inplace=True)
# Compute mutual information between residuals and other features
mutual_info = mutual_info_regression(ethane_test.drop('residuals', axis=1), ethane_test['residuals'])

# Create a DataFrame to pair features with their mutual information scores
mutual_info_df = pd.DataFrame({
    'Feature': ethane_test.drop('residuals', axis=1).columns,  # Feature names
    'Mutual Information': mutual_info  # Corresponding mutual information scores
})

# Sort the DataFrame by mutual information scores in descending order
mutual_info_df = mutual_info_df.sort_values(by='Mutual Information', ascending=False)

# mutual_info_df.drop(15, inplace=True)
# mutual_info_df.drop(16, inplace=True)
# mutual_info_df.drop(17, inplace=True)


# Plot the mutual information scores
plt.figure(figsize=(10, 6))
sns.barplot(x='Mutual Information', y='Feature', data=mutual_info_df, palette='coolwarm')
plt.title('Mutual information between input features and (combined) residuals')
plt.xlabel('Mutual Information')
plt.ylabel('Feature')
plt.show()
print(mutual_info_df)

with open(os.path.join(r"C:\Users\vwgei\Documents\PVOCAL\ensemble\PASTEL_combined\v0_1_5_DMS", f"combined_MI_residuals.pkl"), "wb") as f:
    pickle.dump(mutual_info_df, f)

# Calculate the correlation matrix for all features with respect to residuals
correlation_matrix = ethane_test.corr(method='pearson')

# Extract the correlation values for 'residuals' with all other variables
residuals_corr = correlation_matrix['residuals']

# Sort the correlations to find the most positively or negatively correlated variables
sorted_residuals_corr = residuals_corr.sort_values(ascending=False)

# Print the sorted correlations
print(sorted_residuals_corr)

# Create a heatmap of the correlation matrix (for visualization)
plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title("Correlation Heatmap")
plt.show()

In [ ]:
# # Assuming ethane_test is your dataset
# # Split the dataset into two hemispheres
# north_hemisphere = merged_ethane[merged_ethane['Latitude_0'] > 0]
# south_hemisphere = merged_ethane[merged_ethane['Latitude_0'] < 0]

# # Calculate the average residuals for each hemisphere
# average_residual_north = north_hemisphere['residuals'].mean()
# average_residual_south = south_hemisphere['residuals'].mean()

# # Output the results
# print(f"Average residual in the Northern Hemisphere: {average_residual_north}")
# print(f"Average residual in the Southern Hemisphere: {average_residual_south}")

In [ ]:
ats = "Ethane"

# Open the file in binary read mode and load the object
with open(rf"C:\Users\vwgei\Documents\PVOCAL\ensemble\{ats}_cluster_stats.pkl", 'rb') as file:
    var_cluster_stats = pickle.load(file)

test_mi = []
for i in var_cluster_stats.keys():
    test_mi.append(var_cluster_stats[i]['moran_I_test_RF'])

print(f"Average RF testing Morans I: {np.average(test_mi)}")

input_features = [
    'mean_distance_to_t1000_cities', 'min_distance_to_t1000_cities',
    'Latitude_0', 'Longitude_0', 'AltP_meters_0',
    'Specific_Humidity_0', 'Potential_Temperature_0', 'mean_lcl', 'bearings_from_origin_-24',
    'sum_moisture_flux', 'mean_mixing_depth', 'sum_solar_radiation', 'domain_indicator', 
    'time_difference_seconds_0', 'cos_julian_day', 'hour_cos_0', 
]

var_cluster_stats[1]['sorted_idx']

efid = []
for i in var_cluster_stats.keys():
    efid.append(var_cluster_stats[i]['sorted_idx'])

# Extract the top 3 feature indices across all ensemble members
top_features = []
for entry in efid:
    top_features.extend(entry[-1:])  # Get the last 3 indices

# Convert indices to feature names
top_feature_names = [input_features[i] for i in top_features]

# Count occurrences of each feature
feature_counts = Counter(top_feature_names)

# Sort the feature counts by frequency (value) in descending order
sorted_features = sorted(feature_counts.items(), key=lambda x: x[1], reverse=True)
sorted_feature_names, sorted_counts = zip(*sorted_features)

# Plot sorted histogram as horizontal bars
plt.figure(figsize=(10, 6))
bars = plt.barh(sorted_feature_names, sorted_counts, color='gold')

# Add value labels on bars
for bar, count in zip(bars, sorted_counts):
    plt.text(
        bar.get_width() + 0.1,  # X-coordinate of text (just beyond the bar's length)
        bar.get_y() + bar.get_height() / 2,  # Y-coordinate of text (centered vertically on the bar)
        str(count),  # Text to display
        va='center', color='gray', fontsize=10
    )

plt.gca().invert_yaxis()  # Invert Y-axis to have the most important feature at the top
plt.title(f'Histogram of Top Feature Across {ats} Regional Ensemble')
plt.xlabel('Frequency')
plt.ylabel('Features')
plt.savefig(rf"C:\Users\vwgei\Documents\PVOCAL\ensemble\regional_ensemble\{PASTEL_version}_{ats}\{ats}_re_feature_importances_composite.png", dpi=300, bbox_inches='tight')
plt.tight_layout()
plt.show()

In [ ]:
# Open the file in binary read mode and load the object
with open(rf"C:\Users\vwgei\Documents\PVOCAL\ensemble\global_models\{PASTEL_version}_O3\data\O3_global_X_train.pkl", 'rb') as file:
    o3_train = pickle.load(file)
with open(rf"C:\Users\vwgei\Documents\PVOCAL\ensemble\global_models\{PASTEL_version}_O3\data\O3_global_X_test.pkl", 'rb') as file:
    o3_test = pickle.load(file)

o3_residual_train = np.load(rf"C:\Users\vwgei\Documents\PVOCAL\ensemble\PASTEL_combined\{PASTEL_version}_O3\O3_PASTEL_combined_residuals_train.npy")
o3_residual_test = np.load(rf"C:\Users\vwgei\Documents\PVOCAL\ensemble\PASTEL_combined\{PASTEL_version}_O3\O3_PASTEL_combined_residuals_test.npy")
o3_pred_train = np.load(rf"C:\Users\vwgei\Documents\PVOCAL\ensemble\PASTEL_combined\{PASTEL_version}_O3\O3_PASTEL_combined_pred_train.npy")
o3_pred_test = np.load(rf"C:\Users\vwgei\Documents\PVOCAL\ensemble\PASTEL_combined\{PASTEL_version}_O3\O3_PASTEL_combined_pred_test.npy")

# o3_test['iindex'] = o3_test.index

# merged_o3 = o3_test.merge(df[['Latitude_0', 'Longitude_0']], left_on='iindex', right_index=True, how='left')

# Combine the residuals from both datasets to compute vmin and vmax
all_residuals = o3_residual_test #np.concatenate([dms_residual_train, dms_residual_test])

# Calculate the required statistics
mean_residual = np.mean(all_residuals)
median_residual = np.median(all_residuals)
std_residual = np.std(all_residuals)
above_2sd = np.sum(all_residuals > mean_residual + 2 * std_residual)
below_2sd = np.sum(all_residuals < mean_residual - 2 * std_residual)
n = len(all_residuals)

# Set up the color normalization (using linear scale)
norm = mcolors.Normalize(vmin=mean_residual - 2 * std_residual, vmax=mean_residual + 2 * std_residual)

# Create the plot
fig, ax = plt.subplots(figsize=(12, 8), subplot_kw={'projection': ccrs.PlateCarree(central_longitude=240)})
ax.set_extent([-136, 120, -90, 90], crs=ccrs.PlateCarree(central_longitude=240))

# Add basemap features
ax.coastlines(resolution='50m')
ax.add_feature(cfeature.BORDERS.with_scale('50m'), linestyle=':')
ax.add_feature(cfeature.LAND.with_scale('50m'), facecolor='white')
ax.add_feature(cfeature.OCEAN.with_scale('50m'), facecolor='white')

# Add gridlines
gl = ax.gridlines(draw_labels=True, linestyle="--", color="gray")
gl.top_labels = gl.right_labels = False  # optional: turn off top/right labels for cleaner look

# # Plot the data for the training set
# sc0 = ax.scatter(
#     dms_train['Longitude_0'].values, 
#     dms_train['Latitude_0'].values, 
#     c=dms_residual_train,  # Use numeric residuals for consistent coloring
#     cmap='coolwarm_r', 
#     s=20, 
#     transform=ccrs.PlateCarree(),
#     norm=norm  # Apply the same normalization
# )

# Plot the data for the test set
sc1 = ax.scatter(
    o3_test['Longitude_0'].values, 
    o3_test['Latitude_0'].values, 
    c=o3_residual_test,  # Use numeric residuals for consistent coloring
    cmap='coolwarm_r', 
    s=20, 
    transform=ccrs.PlateCarree(),
    norm=norm  # Apply the same normalization
)

# Add the shared colorbar
cbar = plt.colorbar(sc1, ax=ax, orientation='vertical', pad=0.05, aspect=30, shrink=0.7, extend='both')
cbar.set_label(f'O3 (ppbv)')

# Title and labels
ax.set_title(f"PASTEL Residuals\n[Test set]")

# Annotate the statistics in the lower left corner
stats_text = (f"n: {n:.0f}\n"
              f"Median: {median_residual:.2f}\n"
              f"Mean: {mean_residual:.2f}\n"
              f"Std Dev: {std_residual:.2f}\n"
              f"# > 2 Std Dev: {above_2sd}\n"
              f"# < -2 Std Dev: {below_2sd}")
ax.text(-130, -85, stats_text, fontsize=12, color='black', ha='left', va='bottom', backgroundcolor='white')

# Save and display the plot
plt.savefig(rf"C:\Users\vwgei\Documents\PVOCAL\plots\O3\O3_residuals_test.png", dpi=300, bbox_inches='tight')
plt.tight_layout()
plt.show()

# Combine the residuals from both datasets to compute vmin and vmax
all_pred = o3_pred_test#np.concatenate([dms_pred_train, dms_pred_test])

# Calculate the required statistics
mean_pred = np.mean(all_pred)
median_pred = np.median(all_pred)
std_pred = np.std(all_pred)
above_2sd = np.sum(all_pred > mean_pred + 2 * std_pred)
below_2sd = np.sum(all_pred < mean_pred - 2 * std_pred)
n = len(all_pred)

min = np.min(all_pred)
max = np.max(all_pred)


norm = mcolors.LogNorm(vmin=np.min(o3_pred_test), vmax=np.max(o3_pred_test))

# Create the plot
fig, ax = plt.subplots(figsize=(12, 8), subplot_kw={'projection': ccrs.PlateCarree(central_longitude=240)})
ax.set_extent([-136, 120, -90, 90], crs=ccrs.PlateCarree(central_longitude=240))

# Add basemap features
ax.coastlines(resolution='50m')
ax.add_feature(cfeature.BORDERS.with_scale('50m'), linestyle=':')
ax.add_feature(cfeature.LAND.with_scale('50m'), facecolor='white')
ax.add_feature(cfeature.OCEAN.with_scale('50m'), facecolor='white')

# Add gridlines
gl = ax.gridlines(draw_labels=True, linestyle="--", color="gray")
gl.top_labels = gl.right_labels = False  # optional: turn off top/right labels for cleaner look

# # Plot the data for the training set
# sc0 = ax.scatter(
#     dms_train['Longitude_0'].values, 
#     dms_train['Latitude_0'].values, 
#     c=dms_pred_train,  # Use numeric residuals for consistent coloring
#     cmap='viridis', 
#     s=20, 
#     transform=ccrs.PlateCarree(),
#     norm=norm  # Apply the same normalization
# )

# Plot the data for the test set
sc1 = ax.scatter(
    o3_test['Longitude_0'].values, 
    o3_test['Latitude_0'].values, 
    c=o3_pred_test,  # Use numeric residuals for consistent coloring
    cmap='viridis', 
    s=20, 
    transform=ccrs.PlateCarree(),
    norm=norm  # Apply the same normalization
)

# Add the shared colorbar
cbar = plt.colorbar(sc1, ax=ax, orientation='vertical', pad=0.05, aspect=30, shrink=0.7,) #extend='both')
cbar.set_label(f'O3 (ppbv)')

# Title and labels
ax.set_title(f"PASTEL Predicted O3\n[Test Set]")

# Annotate the statistics in the lower left corner
stats_text = (f"n: {n:.0f}\n"
              f"Median: {median_pred:.2f}\n"
              f"Mean: {mean_pred:.2f}\n"
              f"Std Dev: {std_pred:.2f}\n"
              f"# > 2 Std Dev: {above_2sd}\n"
            #   f"< -2 Std Dev: {below_2sd}"
            )
ax.text(-130, -85, stats_text, fontsize=12, color='black', ha='left', va='bottom', backgroundcolor='white')

# Save and display the plot
plt.savefig(rf"C:\Users\vwgei\Documents\PVOCAL\plots\O3\O3_combined_pred_test.png", dpi=300, bbox_inches='tight')
plt.tight_layout()
plt.show()

In [ ]:
o3_snr = np.var(o3_pred_test) / np.var(o3_residual_test)
o3_snr

In [ ]:
from sklearn.feature_selection import mutual_info_regression
o3_test['residuals'] = o3_residual_test
o3_test.dropna(inplace=True)
# Compute mutual information between residuals and other features
mutual_info = mutual_info_regression(o3_test.drop('residuals', axis=1), o3_test['residuals'])

# Create a DataFrame to pair features with their mutual information scores
mutual_info_df = pd.DataFrame({
    'Feature': o3_test.drop('residuals', axis=1).columns,  # Feature names
    'Mutual Information': mutual_info  # Corresponding mutual information scores
})

# Sort the DataFrame by mutual information scores in descending order
mutual_info_df = mutual_info_df.sort_values(by='Mutual Information', ascending=False)

# Plot the mutual information scores
plt.figure(figsize=(10, 6))
sns.barplot(x='Mutual Information', y='Feature', data=mutual_info_df, palette='coolwarm')
plt.title('Mutual Information between Features and Residuals')
plt.xlabel('Mutual Information')
plt.ylabel('Feature')
plt.show()
print(mutual_info_df)

with open(os.path.join(rf"C:\Users\vwgei\Documents\PVOCAL\ensemble\PASTEL_combined\{PASTEL_version}_O3", f"combined_MI_residuals.pkl"), "wb") as f:
    pickle.dump(mutual_info_df, f)

# Calculate the correlation matrix for all features with respect to residuals
correlation_matrix = o3_test.corr(method='pearson')

# Extract the correlation values for 'residuals' with all other variables
residuals_corr = correlation_matrix['residuals']

# Sort the correlations to find the most positively or negatively correlated variables
sorted_residuals_corr = residuals_corr.sort_values(ascending=False)

# Print the sorted correlations
print(sorted_residuals_corr)

# Create a heatmap of the correlation matrix (for visualization)
plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title("Correlation Heatmap")
plt.show()

In [ ]:
ats = "O3"

# Open the file in binary read mode and load the object
with open(rf"C:\Users\vwgei\Documents\PVOCAL\ensemble\{ats}_cluster_stats.pkl", 'rb') as file:
    var_cluster_stats = pickle.load(file)

test_mi = []
for i in var_cluster_stats.keys():
    test_mi.append(var_cluster_stats[i]['moran_I_test_RF'])

print(f"Average RF testing Morans I: {np.average(test_mi)}")

input_features = [
    'mean_distance_to_t1000_cities', 'min_distance_to_t1000_cities',
    'Latitude_0', 'Longitude_0', 'AltP_meters_0',
    'Specific_Humidity_0', 'Potential_Temperature_0', 'mean_lcl', 'bearings_from_origin_-24',
    'sum_moisture_flux', 'mean_mixing_depth', 'sum_solar_radiation', 'domain_indicator', 
    'time_difference_seconds_0', 'cos_julian_day', 'hour_cos_0', 
]

var_cluster_stats[1]['sorted_idx']

efid = []
for i in var_cluster_stats.keys():
    efid.append(var_cluster_stats[i]['sorted_idx'])

# Extract the top 3 feature indices across all ensemble members
top_features = []
for entry in efid:
    top_features.extend(entry[-1:])  # Get the last 3 indices

# Convert indices to feature names
top_feature_names = [input_features[i] for i in top_features]

# Count occurrences of each feature
feature_counts = Counter(top_feature_names)

# Sort the feature counts by frequency (value) in descending order
sorted_features = sorted(feature_counts.items(), key=lambda x: x[1], reverse=True)
sorted_feature_names, sorted_counts = zip(*sorted_features)

# Plot sorted histogram as horizontal bars
plt.figure(figsize=(10, 6))
bars = plt.barh(sorted_feature_names, sorted_counts, color='yellow')

# Add value labels on bars
for bar, count in zip(bars, sorted_counts):
    plt.text(
        bar.get_width() + 0.1,  # X-coordinate of text (just beyond the bar's length)
        bar.get_y() + bar.get_height() / 2,  # Y-coordinate of text (centered vertically on the bar)
        str(count),  # Text to display
        va='center', color='gray', fontsize=10
    )

plt.gca().invert_yaxis()  # Invert Y-axis to have the most important feature at the top
plt.title(f'Histogram of Top Feature Across {ats} Regional Ensemble')
plt.xlabel('Frequency')
plt.ylabel('Features')
plt.savefig(rf"C:\Users\vwgei\Documents\PVOCAL\ensemble\regional_ensemble\{PASTEL_version}_{ats}\{ats}_re_feature_importances_composite.png", dpi=300, bbox_inches='tight')
plt.tight_layout()
plt.show()

In [ ]:
# Open the file in binary read mode and load the object
with open(rf"C:\Users\vwgei\Documents\PVOCAL\ensemble\global_models\{PASTEL_version}_CO\data\CO_global_X_train.pkl", 'rb') as file:
    co_train = pickle.load(file)
with open(rf"C:\Users\vwgei\Documents\PVOCAL\ensemble\global_models\{PASTEL_version}_CO\data\CO_global_X_test.pkl", 'rb') as file:
    co_test = pickle.load(file)

co_residual_train = np.load(rf"C:\Users\vwgei\Documents\PVOCAL\ensemble\PASTEL_combined\{PASTEL_version}_CO\CO_PASTEL_combined_residuals_train.npy")
co_residual_test = np.load(rf"C:\Users\vwgei\Documents\PVOCAL\ensemble\PASTEL_combined\{PASTEL_version}_CO\CO_PASTEL_combined_residuals_test.npy")
co_pred_train = np.load(rf"C:\Users\vwgei\Documents\PVOCAL\ensemble\PASTEL_combined\{PASTEL_version}_CO\CO_PASTEL_combined_pred_train.npy")
co_pred_test = np.load(rf"C:\Users\vwgei\Documents\PVOCAL\ensemble\PASTEL_combined\{PASTEL_version}_CO\CO_PASTEL_combined_pred_test.npy")

# co_test['iindex'] = co_test.index

# merged_co = co_test.merge(df[['Latitude_0', 'Longitude_0']], left_on='iindex', right_index=True, how='left')

# Combine the residuals from both datasets to compute vmin and vmax
all_residuals = co_residual_test #np.concatenate([dms_residual_train, dms_residual_test])

# Calculate the required statistics
mean_residual = np.mean(all_residuals)
median_residual = np.median(all_residuals)
std_residual = np.std(all_residuals)
above_2sd = np.sum(all_residuals > mean_residual + 2 * std_residual)
below_2sd = np.sum(all_residuals < mean_residual - 2 * std_residual)
n = len(all_residuals)

# Set up the color normalization (using linear scale)
norm = mcolors.Normalize(vmin=mean_residual - 2 * std_residual, vmax=mean_residual + 2 * std_residual)

# Create the plot
fig, ax = plt.subplots(figsize=(12, 8), subplot_kw={'projection': ccrs.PlateCarree(central_longitude=240)})
ax.set_extent([-136, 120, -90, 90], crs=ccrs.PlateCarree(central_longitude=240))

# Add basemap features
ax.coastlines(resolution='50m')
ax.add_feature(cfeature.BORDERS.with_scale('50m'), linestyle=':')
ax.add_feature(cfeature.LAND.with_scale('50m'), facecolor='white')
ax.add_feature(cfeature.OCEAN.with_scale('50m'), facecolor='white')

# Add gridlines
gl = ax.gridlines(draw_labels=True, linestyle="--", color="gray")
gl.top_labels = gl.right_labels = False  # optional: turn off top/right labels for cleaner look

# # Plot the data for the training set
# sc0 = ax.scatter(
#     dms_train['Longitude_0'].values, 
#     dms_train['Latitude_0'].values, 
#     c=dms_residual_train,  # Use numeric residuals for consistent coloring
#     cmap='coolwarm_r', 
#     s=20, 
#     transform=ccrs.PlateCarree(),
#     norm=norm  # Apply the same normalization
# )

# Plot the data for the test set
sc1 = ax.scatter(
    co_test['Longitude_0'].values, 
    co_test['Latitude_0'].values, 
    c=co_residual_test,  # Use numeric residuals for consistent coloring
    cmap='coolwarm_r', 
    s=20, 
    transform=ccrs.PlateCarree(),
    norm=norm  # Apply the same normalization
)

# Add the shared colorbar
cbar = plt.colorbar(sc1, ax=ax, orientation='vertical', pad=0.05, aspect=30, shrink=0.7, extend='both')
cbar.set_label(f'CO (ppbv)')

# Title and labels
ax.set_title(f"PASTEL Residuals\n[Test set]")

# Annotate the statistics in the lower left corner
stats_text = (f"n: {n:.0f}\n"
              f"Median: {median_residual:.2f}\n"
              f"Mean: {mean_residual:.2f}\n"
              f"Std Dev: {std_residual:.2f}\n"
              f"# > 2 Std Dev: {above_2sd}\n"
              f"# < -2 Std Dev: {below_2sd}")
ax.text(-130, -85, stats_text, fontsize=12, color='black', ha='left', va='bottom', backgroundcolor='white')

# Save and display the plot
plt.savefig(rf"C:\Users\vwgei\Documents\PVOCAL\plots\CO\CO_residuals_test.png", dpi=300, bbox_inches='tight')
plt.tight_layout()
plt.show()

# Combine the residuals from both datasets to compute vmin and vmax
all_pred = co_pred_test#np.concatenate([dms_pred_train, dms_pred_test])

# Calculate the required statistics
mean_pred = np.mean(all_pred)
median_pred = np.median(all_pred)
std_pred = np.std(all_pred)
above_2sd = np.sum(all_pred > mean_pred + 2 * std_pred)
below_2sd = np.sum(all_pred < mean_pred - 2 * std_pred)
n = len(all_pred)

min = np.min(all_pred)
max = np.max(all_pred)


norm = mcolors.LogNorm(vmin=11, vmax=np.max(co_pred_test))

# Create the plot
fig, ax = plt.subplots(figsize=(12, 8), subplot_kw={'projection': ccrs.PlateCarree(central_longitude=240)})
ax.set_extent([-136, 120, -90, 90], crs=ccrs.PlateCarree(central_longitude=240))

# Add basemap features
ax.coastlines(resolution='50m')
ax.add_feature(cfeature.BORDERS.with_scale('50m'), linestyle=':')
ax.add_feature(cfeature.LAND.with_scale('50m'), facecolor='white')
ax.add_feature(cfeature.OCEAN.with_scale('50m'), facecolor='white')

# Add gridlines
gl = ax.gridlines(draw_labels=True, linestyle="--", color="gray")
gl.top_labels = gl.right_labels = False  # optional: turn off top/right labels for cleaner look

# # Plot the data for the training set
# sc0 = ax.scatter(
#     dms_train['Longitude_0'].values, 
#     dms_train['Latitude_0'].values, 
#     c=dms_pred_train,  # Use numeric residuals for consistent coloring
#     cmap='viridis', 
#     s=20, 
#     transform=ccrs.PlateCarree(),
#     norm=norm  # Apply the same normalization
# )

# Plot the data for the test set
sc1 = ax.scatter(
    co_test['Longitude_0'].values, 
    co_test['Latitude_0'].values, 
    c=co_pred_test,  # Use numeric residuals for consistent coloring
    cmap='viridis', 
    s=20, 
    transform=ccrs.PlateCarree(),
    norm=norm  # Apply the same normalization
)

# Add the shared colorbar
cbar = plt.colorbar(sc1, ax=ax, orientation='vertical', pad=0.05, aspect=30, shrink=0.7,) #extend='both')
cbar.set_label(f'CO (ppbv)')

# Title and labels
ax.set_title(f"PASTEL Predicted CO\n[Test Set]")

# Annotate the statistics in the lower left corner
stats_text = (f"n: {n:.0f}\n"
              f"Median: {median_pred:.2f}\n"
              f"Mean: {mean_pred:.2f}\n"
              f"Std Dev: {std_pred:.2f}\n"
              f"# > 2 Std Dev: {above_2sd}\n"
            #   f"< -2 Std Dev: {below_2sd}"
            )
ax.text(-130, -85, stats_text, fontsize=12, color='black', ha='left', va='bottom', backgroundcolor='white')

# Save and display the plot
plt.savefig(rf"C:\Users\vwgei\Documents\PVOCAL\plots\CO\CO_combined_pred_test.png", dpi=300, bbox_inches='tight')
plt.tight_layout()
plt.show()

In [ ]:
co_snr = np.var(co_pred_test) / np.var(co_residual_test)
co_snr

In [ ]:
from sklearn.feature_selection import mutual_info_regression
co_test['residuals'] = co_residual_test
co_test.dropna(inplace=True)
# Compute mutual information between residuals and other features
mutual_info = mutual_info_regression(co_test.drop('residuals', axis=1), co_test['residuals'])

# Create a DataFrame to pair features with their mutual information scores
mutual_info_df = pd.DataFrame({
    'Feature': co_test.drop('residuals', axis=1).columns,  # Feature names
    'Mutual Information': mutual_info  # Corresponding mutual information scores
})

# Sort the DataFrame by mutual information scores in descending order
mutual_info_df = mutual_info_df.sort_values(by='Mutual Information', ascending=False)

# Plot the mutual information scores
plt.figure(figsize=(10, 6))
sns.barplot(x='Mutual Information', y='Feature', data=mutual_info_df, palette='coolwarm')
plt.title('Mutual Information between input features and residuals')
plt.xlabel('Mutual Information')
plt.ylabel('Feature')
plt.show()
print(mutual_info_df)

with open(os.path.join(rf"C:\Users\vwgei\Documents\PVOCAL\ensemble\PASTEL_combined\{PASTEL_version}_CO", f"combined_MI_residuals.pkl"), "wb") as f:
    pickle.dump(mutual_info_df, f)

# Calculate the correlation matrix for all features with respect to residuals
correlation_matrix = co_test.corr(method='pearson')

# Extract the correlation values for 'residuals' with all other variables
residuals_corr = correlation_matrix['residuals']

# Sort the correlations to find the most positively or negatively correlated variables
sorted_residuals_corr = residuals_corr.sort_values(ascending=False)

# Print the sorted correlations
print(sorted_residuals_corr)

# Create a heatmap of the correlation matrix (for visualization)
plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title("Correlation Heatmap")
plt.show()

In [ ]:
ats = "CO"

# Open the file in binary read mode and load the object
with open(rf"C:\Users\vwgei\Documents\PVOCAL\ensemble\{ats}_cluster_stats.pkl", 'rb') as file:
    var_cluster_stats = pickle.load(file)

test_mi = []
for i in var_cluster_stats.keys():
    test_mi.append(var_cluster_stats[i]['moran_I_test_RF'])

print(f"Average RF testing Morans I: {np.average(test_mi)}")

input_features = [
    'mean_distance_to_t1000_cities', 'min_distance_to_t1000_cities',
    'Latitude_0', 'Longitude_0', 'AltP_meters_0',
    'Specific_Humidity_0', 'Potential_Temperature_0', 'mean_lcl', 'bearings_from_origin_-24',
    'sum_moisture_flux', 'mean_mixing_depth', 'sum_solar_radiation', 'domain_indicator', 
    'time_difference_seconds_0', 'cos_julian_day', 'hour_cos_0', 
]

var_cluster_stats[1]['sorted_idx']

efid = []
for i in var_cluster_stats.keys():
    efid.append(var_cluster_stats[i]['sorted_idx'])

# Extract the top 3 feature indices across all ensemble members
top_features = []
for entry in efid:
    top_features.extend(entry[-1:])  # Get the last 3 indices

# Convert indices to feature names
top_feature_names = [input_features[i] for i in top_features]

# Count occurrences of each feature
feature_counts = Counter(top_feature_names)

# Sort the feature counts by frequency (value) in descending order
sorted_features = sorted(feature_counts.items(), key=lambda x: x[1], reverse=True)
sorted_feature_names, sorted_counts = zip(*sorted_features)

# Plot sorted histogram as horizontal bars
plt.figure(figsize=(10, 6))
bars = plt.barh(sorted_feature_names, sorted_counts, color='cyan')

# Add value labels on bars
for bar, count in zip(bars, sorted_counts):
    plt.text(
        bar.get_width() + 0.1,  # X-coordinate of text (just beyond the bar's length)
        bar.get_y() + bar.get_height() / 2,  # Y-coordinate of text (centered vertically on the bar)
        str(count),  # Text to display
        va='center', color='gray', fontsize=10
    )

plt.gca().invert_yaxis()  # Invert Y-axis to have the most important feature at the top
plt.title(f'Histogram of Top Feature Across {ats} Ensemble')
plt.xlabel('Frequency')
plt.ylabel('Features')
plt.savefig(rf"C:\Users\vwgei\Documents\PVOCAL\ensemble\regional_ensemble\{PASTEL_version}_{ats}\{ats}_re_feature_importances_composite.png", dpi=300, bbox_inches='tight')
plt.tight_layout()
plt.show()

In [ ]:
# # Open the file in binary read mode and load the object
with open(rf"C:\Users\vwgei\Documents\PVOCAL\ensemble\global_models\{PASTEL_version}_CH4\data\CH4_global_X_train.pkl", 'rb') as file:
    ch4_train = pickle.load(file)
with open(rf"C:\Users\vwgei\Documents\PVOCAL\ensemble\global_models\{PASTEL_version}_CH4\data\CH4_global_X_test.pkl", 'rb') as file:
    ch4_test = pickle.load(file)

# ch4_residual_train = np.load(r"C:\Users\vwgei\Documents\PVOCAL\ensemble_working\PASTEL_combined\V4_CH4\CH4_PASTEL_combined_residuals_train.npy")
ch4_residual_test = np.load(rf"C:\Users\vwgei\Documents\PVOCAL\ensemble\PASTEL_combined\{PASTEL_version}_CH4\CH4_PASTEL_combined_residuals_test.npy")
# ch4_pred_train = np.load(r"C:\Users\vwgei\Documents\PVOCAL\ensemble_working\PASTEL_combined\V4_CH4\CH4_PASTEL_combined_pred_train.npy")
ch4_pred_test = np.load(rf"C:\Users\vwgei\Documents\PVOCAL\ensemble\PASTEL_combined\{PASTEL_version}_CH4\CH4_PASTEL_combined_pred_test.npy")

# ch4_test['iindex'] = ch4_test.index

# merged_ch4 = ch4_test.merge(df[['Latitude_0', 'Longitude_0']], left_on='iindex', right_index=True, how='left')

# Combine the residuals from both datasets to compute vmin and vmax
all_residuals = ch4_residual_test #np.concatenate([dms_residual_train, dms_residual_test])

# Calculate the required statistics
mean_residual = np.mean(all_residuals)
median_residual = np.median(all_residuals)
std_residual = np.std(all_residuals)
above_2sd = np.sum(all_residuals > mean_residual + 2 * std_residual)
below_2sd = np.sum(all_residuals < mean_residual - 2 * std_residual)
n = len(all_residuals)

# Set up the color normalization (using linear scale)
norm = mcolors.Normalize(vmin=mean_residual - 2 * std_residual, vmax=mean_residual + 2 * std_residual)

# Create the plot
fig, ax = plt.subplots(figsize=(12, 8), subplot_kw={'projection': ccrs.PlateCarree(central_longitude=240)})
ax.set_extent([-136, 120, -90, 90], crs=ccrs.PlateCarree(central_longitude=240))

# Add basemap features
ax.coastlines(resolution='50m')
ax.add_feature(cfeature.BORDERS.with_scale('50m'), linestyle=':')
ax.add_feature(cfeature.LAND.with_scale('50m'), facecolor='white')
ax.add_feature(cfeature.OCEAN.with_scale('50m'), facecolor='white')

# Add gridlines
gl = ax.gridlines(draw_labels=True, linestyle="--", color="gray")
gl.top_labels = gl.right_labels = False  # optional: turn off top/right labels for cleaner look

# # Plot the data for the training set
# sc0 = ax.scatter(
#     dms_train['Longitude_0'].values, 
#     dms_train['Latitude_0'].values, 
#     c=dms_residual_train,  # Use numeric residuals for consistent coloring
#     cmap='coolwarm_r', 
#     s=20, 
#     transform=ccrs.PlateCarree(),
#     norm=norm  # Apply the same normalization
# )

# Plot the data for the test set
sc1 = ax.scatter(
    ch4_test['Longitude_0'].values, 
    ch4_test['Latitude_0'].values, 
    c=ch4_residual_test,  # Use numeric residuals for consistent coloring
    cmap='coolwarm_r', 
    s=20, 
    transform=ccrs.PlateCarree(),
    norm=norm  # Apply the same normalization
)

# Add the shared colorbar
cbar = plt.colorbar(sc1, ax=ax, orientation='vertical', pad=0.05, aspect=30, shrink=0.7, extend='both')
cbar.set_label(f'CH4 (ppbv)')

# Title and labels
ax.set_title(f"PASTEL Residuals\n[Test set]")

# Annotate the statistics in the lower left corner
stats_text = (f"n: {n:.0f}\n"
              f"Median: {median_residual:.2f}\n"
              f"Mean: {mean_residual:.2f}\n"
              f"Std Dev: {std_residual:.2f}\n"
              f"# > 2 Std Dev: {above_2sd}\n"
              f"# < -2 Std Dev: {below_2sd}")
ax.text(-130, -85, stats_text, fontsize=12, color='black', ha='left', va='bottom', backgroundcolor='white')

# Save and display the plot
plt.savefig(rf"C:\Users\vwgei\Documents\PVOCAL\plots\CH4\CH4_residuals_test.png", dpi=300, bbox_inches='tight')
plt.tight_layout()
plt.show()

# Combine the residuals from both datasets to compute vmin and vmax
all_pred = ch4_pred_test#np.concatenate([dms_pred_train, dms_pred_test])

# Calculate the required statistics
mean_pred = np.mean(all_pred)
median_pred = np.median(all_pred)
std_pred = np.std(all_pred)
above_2sd = np.sum(all_pred > mean_pred + 2 * std_pred)
below_2sd = np.sum(all_pred < mean_pred - 2 * std_pred)
n = len(all_pred)

min = np.min(all_pred)
max = np.max(all_pred)


norm = mcolors.LogNorm(vmin=np.min(ch4_pred_test), vmax=np.max(ch4_pred_test))


# Create the plot
fig, ax = plt.subplots(figsize=(12, 8), subplot_kw={'projection': ccrs.PlateCarree(central_longitude=240)})
ax.set_extent([-136, 120, -90, 90], crs=ccrs.PlateCarree(central_longitude=240))

# Add basemap features
ax.coastlines(resolution='50m')
ax.add_feature(cfeature.BORDERS.with_scale('50m'), linestyle=':')
ax.add_feature(cfeature.LAND.with_scale('50m'), facecolor='white')
ax.add_feature(cfeature.OCEAN.with_scale('50m'), facecolor='white')

# Add gridlines
gl = ax.gridlines(draw_labels=True, linestyle="--", color="gray")
gl.top_labels = gl.right_labels = False  # optional: turn off top/right labels for cleaner look

# # Plot the data for the training set
# sc0 = ax.scatter(
#     dms_train['Longitude_0'].values, 
#     dms_train['Latitude_0'].values, 
#     c=dms_pred_train,  # Use numeric residuals for consistent coloring
#     cmap='viridis', 
#     s=20, 
#     transform=ccrs.PlateCarree(),
#     norm=norm  # Apply the same normalization
# )

# Plot the data for the test set
sc1 = ax.scatter(
    ch4_test['Longitude_0'].values, 
    ch4_test['Latitude_0'].values, 
    c=ch4_pred_test,  # Use numeric residuals for consistent coloring
    cmap='viridis', 
    s=20, 
    transform=ccrs.PlateCarree(),
    norm=norm  # Apply the same normalization
)

# Add the shared colorbar
cbar = plt.colorbar(sc1, ax=ax, orientation='vertical', pad=0.05, aspect=30, shrink=0.7,) #extend='both')
cbar.set_label(f'CH4 (ppbv)')

# Title and labels
ax.set_title(f"PASTEL Predicted CH4\n[Test Set]")

# Annotate the statistics in the lower left corner
stats_text = (f"n: {n:.0f}\n"
              f"Median: {median_pred:.2f}\n"
              f"Mean: {mean_pred:.2f}\n"
              f"Std Dev: {std_pred:.2f}\n"
              f"# > 2 Std Dev: {above_2sd}\n"
            #   f"< -2 Std Dev: {below_2sd}"
            )
ax.text(-130, -85, stats_text, fontsize=12, color='black', ha='left', va='bottom', backgroundcolor='white')

# Save and display the plot
plt.savefig(rf"C:\Users\vwgei\Documents\PVOCAL\plots\CH4\CH4_combined_pred_test.png", dpi=300, bbox_inches='tight')
plt.tight_layout()
plt.show()

In [ ]:
ch4_snr = np.var(ch4_pred_test) / np.var(ch4_residual_test)
ch4_snr

In [ ]:
from sklearn.feature_selection import mutual_info_regression
ch4_test['residuals'] = ch4_residual_test
ch4_test.dropna(inplace=True)
# Compute mutual information between residuals and other features
mutual_info = mutual_info_regression(ch4_test.drop('residuals', axis=1), ch4_test['residuals'])

# Create a DataFrame to pair features with their mutual information scores
mutual_info_df = pd.DataFrame({
    'Feature': ch4_test.drop('residuals', axis=1).columns,  # Feature names
    'Mutual Information': mutual_info  # Corresponding mutual information scores
})

# Sort the DataFrame by mutual information scores in descending order
mutual_info_df = mutual_info_df.sort_values(by='Mutual Information', ascending=False)

# Plot the mutual information scores
plt.figure(figsize=(10, 6))
sns.barplot(x='Mutual Information', y='Feature', data=mutual_info_df, palette='coolwarm')
plt.title('Mutual Information between Features and Residuals')
plt.xlabel('Mutual Information')
plt.ylabel('Feature')
plt.show()
print(mutual_info_df)

with open(os.path.join(rf"C:\Users\vwgei\Documents\PVOCAL\ensemble\PASTEL_combined\{PASTEL_version}_CH4", f"combined_MI_residuals.pkl"), "wb") as f:
    pickle.dump(mutual_info_df, f)

# Calculate the correlation matrix for all features with respect to residuals
correlation_matrix = ch4_test.corr(method='pearson')

# Extract the correlation values for 'residuals' with all other variables
residuals_corr = correlation_matrix['residuals']

# Sort the correlations to find the most positively or negatively correlated variables
sorted_residuals_corr = residuals_corr.sort_values(ascending=False)

# Print the sorted correlations
print(sorted_residuals_corr)

# Create a heatmap of the correlation matrix (for visualization)
plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title("Correlation Heatmap")
plt.show()

In [ ]:
ats = "CH4"

# Open the file in binary read mode and load the object
with open(rf"C:\Users\vwgei\Documents\PVOCAL\ensemble\{ats}_cluster_stats.pkl", 'rb') as file:
    var_cluster_stats = pickle.load(file)

test_mi = []
for i in var_cluster_stats.keys():
    test_mi.append(var_cluster_stats[i]['moran_I_test_RF'])

print(f"Average RF testing Morans I: {np.nanmean(test_mi)}")

input_features = [
    'mean_distance_to_t1000_cities', 'min_distance_to_t1000_cities',
    'Latitude_0', 'Longitude_0', 'AltP_meters_0',
    'Specific_Humidity_0', 'Potential_Temperature_0', 'mean_lcl', 'bearings_from_origin_-24',
    'sum_moisture_flux', 'mean_mixing_depth', 'sum_solar_radiation', 'domain_indicator', 
    'time_difference_seconds_0', 'cos_julian_day', 'hour_cos_0', 
]

var_cluster_stats[1]['sorted_idx']

efid = []
for i in var_cluster_stats.keys():
    efid.append(var_cluster_stats[i]['sorted_idx'])

# Extract the top 3 feature indices across all ensemble members
top_features = []
for entry in efid:
    top_features.extend(entry[-1:])  # Get the last 3 indices

# Convert indices to feature names
top_feature_names = [input_features[i] for i in top_features]

# Count occurrences of each feature
feature_counts = Counter(top_feature_names)

# Sort the feature counts by frequency (value) in descending order
sorted_features = sorted(feature_counts.items(), key=lambda x: x[1], reverse=True)
sorted_feature_names, sorted_counts = zip(*sorted_features)

# Plot sorted histogram as horizontal bars
plt.figure(figsize=(10, 6))
bars = plt.barh(sorted_feature_names, sorted_counts, color='blue')

# Add value labels on bars
for bar, count in zip(bars, sorted_counts):
    plt.text(
        bar.get_width() + 0.1,  # X-coordinate of text (just beyond the bar's length)
        bar.get_y() + bar.get_height() / 2,  # Y-coordinate of text (centered vertically on the bar)
        str(count),  # Text to display
        va='center', color='gray', fontsize=10
    )

plt.gca().invert_yaxis()  # Invert Y-axis to have the most important feature at the top
plt.title(f'Histogram of Top Feature Across {ats} Ensemble')
plt.xlabel('Frequency')
plt.ylabel('Features')
plt.savefig(rf"C:\Users\vwgei\Documents\PVOCAL\ensemble\regional_ensemble\{PASTEL_version}_{ats}\{ats}_re_feature_importances_composite.png", dpi=300, bbox_inches='tight')
plt.tight_layout()
plt.show()

In [ ]:
from matplotlib.colors import ListedColormap

# Extract the colors from tab20b
tab20_colors = plt.cm.tab20b(np.linspace(0, 1, 20))

# Add a 21st color by interpolating (or define a custom color explicitly)
extra_color = np.array([[0.5, 0.5, 0.5, 1.0]])  # A neutral gray as an example
# extra_color = np.array([[0.0, 1.0, 1.0, 1.0]])  # Cyan: full green, full blue, no red
custom_colors = np.vstack([tab20_colors, extra_color])

# Create a custom ListedColormap
custom_tab20_21 = ListedColormap(custom_colors, name='custom_tab20_21')


# Example dictionary mapping numeric labels to custom string labels
label_mapping = {
    0: "ATom",
    1: "ACCLIP",
    2: "DC3",
    3: "DISCOVER-AQ_FRAPPE",
    4: "FIREX-AQ",
    5: "INTEX-A",
    6: "INTEX-B (C130)",
    7: "INTEX-B (DC8)",
    8: "KORUS-AQ",
    9: "PEM TROPICS A (DC8)",
    10: "PEM TROPICS A (P3)",
    11: "PEM TROPICS B (DC8)",
    12: "PEM TROPICS B (P3)",
    13: "PEM WEST A",
    14: "PEM WEST B",
    15: "SEAC4RS (DC8)",
    16: "TRACE A (DC8)",
    17: "TRACE A (P3)",
    18: "TRACE P (DC8)",
    19: "TRACE P (P3)",
    20: "WINTER",
    # Add more mappings as needed
}


# Drop any rows with missing CH4 data
mtest = df.dropna(subset="CH4")

# Ensure 'Datetime' is a datetime object
mtest['Datetime'] = pd.to_datetime(mtest['Datetime'])

# Extract the year and month to calculate yearly averages
mtest['Year'] = mtest['Datetime'].dt.year

# Group by 'Year' and calculate the mean of 'CH4' for each year
# yearly_avg = mtest.groupby('Year')['CH4'].mean().reset_index()
yearly_percentile = mtest.groupby('Year')['CH4'].quantile(0.10).reset_index()

# Step 1: Convert Yearly Averages to Datetime (Set date to the middle of the year)
yearly_percentile['Datetime'] = pd.to_datetime(yearly_percentile['Year'].astype(str) + '-06-01')

# Step 2: Plot data and yearly averages
plt.figure(figsize=(10, 6))

# Scatter plot of the data
sc = plt.scatter(df['Datetime'], df['CH4'], c=df['domain_indicator'].values,  # Use numeric values for consistent coloring
    cmap=custom_tab20_21, alpha=1, edgecolor='none')

# Step 3: Plot the yearly averages as datetime points
plt.scatter(yearly_percentile['Datetime'], yearly_percentile['CH4'], color='red', label='Yearly Average', zorder=5)

# Step 4: Annotate only the first and last yearly averages
first_point = yearly_percentile.iloc[0]
last_point = yearly_percentile.iloc[-1]

# Annotating the first point
plt.annotate(f"{first_point['CH4']:.0f}",  # Display CH4 value with two decimal places
             (first_point['Datetime'], first_point['CH4']),  # Position the annotation at the point
             textcoords="offset points",  # Position relative to the point
             xytext=(0, 50),  # Slightly offset the label
             ha='center',  # Horizontal alignment to center
             fontsize=9,  # Font size of the label
             color='black')  # Color of the label

# Annotating the last point
plt.annotate(f"{last_point['CH4']:.0f}",  # Display CH4 value with two decimal places
             (last_point['Datetime'], last_point['CH4']),  # Position the annotation at the point
             textcoords="offset points",  # Position relative to the point
             xytext=(0, 50),  # Slightly offset the label
             ha='center',  # Horizontal alignment to center
             fontsize=9,  # Font size of the label
             color='black')  # Color of the label

# Update the colorbar to match the new number of labels
cbar = plt.colorbar(sc, orientation='vertical', pad=0.05, aspect=30, shrink=0.7)
cbar.set_label("Campaign Name")
cbar.set_ticks(range(len(label_mapping)))
cbar.set_ticklabels([label_mapping[i] for i in range(len(label_mapping))])
cbar.ax.invert_yaxis()  # Flip the colorbar to align labels properly


# Step 5: Customize the plot
plt. title('CH4 Mixing Ratios Over Input Dataset', fontsize=16)
plt.xlabel('Datetime', fontsize=14)
plt.ylabel('CH4 (ppbv)', fontsize=14)
plt.grid(True, linestyle='--', alpha=0.6)
plt.xticks(rotation=45, fontsize=10)
plt.legend(fontsize=12)
plt.tight_layout()

plt.savefig(r"C:\Users\vwgei\Documents\PVOCAL\plots\standalone\ch4_timeseries.png", dpi=300, bbox_inches='tight')

# Step 6: Show the plot
plt.show()

In [ ]:
import matplotlib.ticker as ticker
# Open the file in binary read mode and load the object
with open(rf"C:\Users\vwgei\Documents\PVOCAL\ensemble\global_models\{PASTEL_version}_CH3Br\data\CH3Br_global_X_train.pkl", 'rb') as file:
    ch3br_train = pickle.load(file)
with open(rf"C:\Users\vwgei\Documents\PVOCAL\ensemble\global_models\{PASTEL_version}_CH3Br\data\CH3Br_global_X_test.pkl", 'rb') as file:
    ch3br_test = pickle.load(file)

# ch3br_residual_train = np.load(r"C:\Users\vwgei\Documents\PVOCAL\ensemble\PASTEL_combined\V4_CH3Br\CH3Br_PASTEL_combined_residuals_train.npy")
ch3br_residual_test = np.load(rf"C:\Users\vwgei\Documents\PVOCAL\ensemble\PASTEL_combined\{PASTEL_version}_CH3Br\CH3Br_PASTEL_combined_residuals_test.npy")
# ch3br_pred_train = np.load(r"C:\Users\vwgei\Documents\PVOCAL\ensemble\PASTEL_combined\V4_CH3Br\CH3Br_PASTEL_combined_pred_train.npy")
ch3br_pred_test = np.load(rf"C:\Users\vwgei\Documents\PVOCAL\ensemble\PASTEL_combined\{PASTEL_version}_CH3Br\CH3Br_PASTEL_combined_pred_test.npy")

# ch3br_test['iindex'] = ch3br_test.index

# merged_ch3br = ch3br_test.merge(df[['Latitude_0', 'Longitude_0']], left_on='iindex', right_index=True, how='left')

# Combine the residuals from both datasets to compute vmin and vmax
all_residuals = ch3br_residual_test #np.concatenate([dms_residual_train, dms_residual_test])

# Calculate the required statistics
mean_residual = np.mean(all_residuals)
median_residual = np.median(all_residuals)
std_residual = np.std(all_residuals)
above_2sd = np.sum(all_residuals > mean_residual + 2 * std_residual)
below_2sd = np.sum(all_residuals < mean_residual - 2 * std_residual)
n = len(all_residuals)

# Set up the color normalization (using linear scale)
norm = mcolors.Normalize(vmin=mean_residual - 2 * std_residual, vmax=mean_residual + 2 * std_residual)

# Create the plot
fig, ax = plt.subplots(figsize=(12, 8), subplot_kw={'projection': ccrs.PlateCarree(central_longitude=240)})
ax.set_extent([-136, 120, -90, 90], crs=ccrs.PlateCarree(central_longitude=240))

# Add basemap features
ax.coastlines(resolution='50m')
ax.add_feature(cfeature.BORDERS.with_scale('50m'), linestyle=':')
ax.add_feature(cfeature.LAND.with_scale('50m'), facecolor='white')
ax.add_feature(cfeature.OCEAN.with_scale('50m'), facecolor='white')

# Add gridlines
gl = ax.gridlines(draw_labels=True, linestyle="--", color="gray")
gl.top_labels = gl.right_labels = False  # optional: turn off top/right labels for cleaner look

# # Plot the data for the training set
# sc0 = ax.scatter(
#     dms_train['Longitude_0'].values, 
#     dms_train['Latitude_0'].values, 
#     c=dms_residual_train,  # Use numeric residuals for consistent coloring
#     cmap='coolwarm_r', 
#     s=20, 
#     transform=ccrs.PlateCarree(),
#     norm=norm  # Apply the same normalization
# )

# Plot the data for the test set
sc1 = ax.scatter(
    ch3br_test['Longitude_0'].values, 
    ch3br_test['Latitude_0'].values, 
    c=ch3br_residual_test,  # Use numeric residuals for consistent coloring
    cmap='coolwarm_r', 
    s=20, 
    transform=ccrs.PlateCarree(),
    norm=norm  # Apply the same normalization
)

# Add the shared colorbar
cbar = plt.colorbar(sc1, ax=ax, orientation='vertical', pad=0.05, aspect=30, shrink=0.7, extend='both')
cbar.set_label(f'CH3Br (pptv)')

# Title and labels
ax.set_title(f"PASTEL Residuals\n[Test set]")

# Annotate the statistics in the lower left corner
stats_text = (f"n: {n:.0f}\n"
              f"Median: {median_residual:.2f}\n"
              f"Mean: {mean_residual:.2f}\n"
              f"Std Dev: {std_residual:.2f}\n"
              f"# > 2 Std Dev: {above_2sd}\n"
              f"# < -2 Std Dev: {below_2sd}")
ax.text(-130, -85, stats_text, fontsize=12, color='black', ha='left', va='bottom', backgroundcolor='white')

# Save and display the plot
plt.savefig(rf"C:\Users\vwgei\Documents\PVOCAL\plots\CH3Br\CH3Br_residuals_test.png", dpi=300, bbox_inches='tight')
plt.tight_layout()
plt.show()

# Combine the residuals from both datasets to compute vmin and vmax
all_pred = ch3br_pred_test#np.concatenate([dms_pred_train, dms_pred_test])

# Calculate the required statistics
mean_pred = np.mean(all_pred)
median_pred = np.median(all_pred)
std_pred = np.std(all_pred)
above_2sd = np.sum(all_pred > mean_pred + 2 * std_pred)
below_2sd = np.sum(all_pred < mean_pred - 2 * std_pred)
n = len(all_pred)

min = np.min(all_pred)
max = np.max(all_pred)


norm = mcolors.LogNorm(vmin=np.min(ch3br_pred_test), vmax=np.max(ch3br_pred_test))


# Create the plot
fig, ax = plt.subplots(figsize=(12, 8), subplot_kw={'projection': ccrs.PlateCarree(central_longitude=240)})
ax.set_extent([-136, 120, -90, 90], crs=ccrs.PlateCarree(central_longitude=240))

# Add basemap features
ax.coastlines(resolution='50m')
ax.add_feature(cfeature.BORDERS.with_scale('50m'), linestyle=':')
ax.add_feature(cfeature.LAND.with_scale('50m'), facecolor='white')
ax.add_feature(cfeature.OCEAN.with_scale('50m'), facecolor='white')

# Add gridlines
gl = ax.gridlines(draw_labels=True, linestyle="--", color="gray")
gl.top_labels = gl.right_labels = False  # optional: turn off top/right labels for cleaner look

# # Plot the data for the training set
# sc0 = ax.scatter(
#     dms_train['Longitude_0'].values, 
#     dms_train['Latitude_0'].values, 
#     c=dms_pred_train,  # Use numeric residuals for consistent coloring
#     cmap='viridis', 
#     s=20, 
#     transform=ccrs.PlateCarree(),
#     norm=norm  # Apply the same normalization
# )

# Plot the data for the test set
sc1 = ax.scatter(
    ch3br_test['Longitude_0'].values, 
    ch3br_test['Latitude_0'].values, 
    c=ch3br_pred_test,  # Use numeric residuals for consistent coloring
    cmap='viridis', 
    s=20, 
    transform=ccrs.PlateCarree(),
    norm=norm  # Apply the same normalization
)

# Add the shared colorbar
cbar = plt.colorbar(sc1, ax=ax, orientation='vertical', pad=0.05, aspect=30, shrink=0.7,) #extend='both')
cbar.set_label(f'CH3Br (pptv)')

# Customize the ticks for a LogNorm colorbar
cbar.set_ticks(ticker.LogLocator(base=10.0, subs=np.arange(1.0, 10.0)*0.1, numticks=10), update=True)  # Minor ticks
cbar.minorticks_on()  # Ensure minor ticks are shown
cbar.ax.yaxis.set_minor_formatter(ticker.LogFormatter(base=10.0, labelOnlyBase=False))  # Label minor ticks
cbar.ax.yaxis.set_major_formatter(ticker.LogFormatter(base=10.0))  # Default major tick formatting

# Title and labels
ax.set_title(f"PASTEL Predicted CH3Br\n[Test Set]")

# Annotate the statistics in the lower left corner
stats_text = (f"n: {n:.0f}\n"
              f"Median: {median_pred:.2f}\n"
              f"Mean: {mean_pred:.2f}\n"
              f"Std Dev: {std_pred:.2f}\n"
              f"# > 2 Std Dev: {above_2sd}\n"
            #   f"< -2 Std Dev: {below_2sd}"
            )
ax.text(-130, -85, stats_text, fontsize=12, color='black', ha='left', va='bottom', backgroundcolor='white')

# Save and display the plot
plt.savefig(rf"C:\Users\vwgei\Documents\PVOCAL\plots\CH3Br\CH3Br_combined_pred_test.png", dpi=300, bbox_inches='tight')
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.feature_selection import mutual_info_regression
ch3br_test['residuals'] = ch3br_residual_test
ch3br_test.dropna(inplace=True)
# Compute mutual information between residuals and other features
mutual_info = mutual_info_regression(ch3br_test.drop('residuals', axis=1), ch3br_test['residuals'])

# Create a DataFrame to pair features with their mutual information scores
mutual_info_df = pd.DataFrame({
    'Feature': ch3br_test.drop('residuals', axis=1).columns,  # Feature names
    'Mutual Information': mutual_info  # Corresponding mutual information scores
})

# Sort the DataFrame by mutual information scores in descending order
mutual_info_df = mutual_info_df.sort_values(by='Mutual Information', ascending=False)

# Plot the mutual information scores
plt.figure(figsize=(10, 6))
sns.barplot(x='Mutual Information', y='Feature', data=mutual_info_df, palette='coolwarm')
plt.title('Mutual Information between input features and residuals (combined model)')
plt.xlabel('Mutual Information')
plt.ylabel('Feature')
plt.show()
print(mutual_info_df)

with open(os.path.join(rf"C:\Users\vwgei\Documents\PVOCAL\ensemble\PASTEL_combined\{PASTEL_version}_CH3Br", f"combined_MI_residuals.pkl"), "wb") as f:
    pickle.dump(mutual_info_df, f)

# Calculate the correlation matrix for all features with respect to residuals
correlation_matrix = ch3br_test.corr(method='pearson')

# Extract the correlation values for 'residuals' with all other variables
residuals_corr = correlation_matrix['residuals']

# Sort the correlations to find the most positively or negatively correlated variables
sorted_residuals_corr = residuals_corr.sort_values(ascending=False)

# Print the sorted correlations
print(sorted_residuals_corr)

# Create a heatmap of the correlation matrix (for visualization)
plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title("Correlation Heatmap")
plt.show()

In [ ]:
ch3br_snr = np.var(ch3br_pred_test) / np.var(ch3br_residual_test)
ch3br_snr

In [ ]:
ats = "CH3Br"

# Open the file in binary read mode and load the object
with open(rf"C:\Users\vwgei\Documents\PVOCAL\ensemble\{ats}_cluster_stats.pkl", 'rb') as file:
    var_cluster_stats = pickle.load(file)

test_mi = []
for i in var_cluster_stats.keys():
    test_mi.append(var_cluster_stats[i]['moran_I_test_RF'])

print(f"Average RF testing Morans I: {np.average(test_mi)}")

input_features = [
    'mean_distance_to_t1000_cities', 'min_distance_to_t1000_cities',
    'Latitude_0', 'Longitude_0', 'AltP_meters_0',
    'Specific_Humidity_0', 'Potential_Temperature_0', 'mean_lcl', 'bearings_from_origin_-24',
    'sum_moisture_flux', 'mean_mixing_depth', 'sum_solar_radiation', 'domain_indicator', 
    'time_difference_seconds_0', 'cos_julian_day', 'hour_cos_0', 
]

var_cluster_stats[1]['sorted_idx']

efid = []
for i in var_cluster_stats.keys():
    efid.append(var_cluster_stats[i]['sorted_idx'])

# Extract the top 3 feature indices across all ensemble members
top_features = []
for entry in efid:
    top_features.extend(entry[-1:])  # Get the last 3 indices

# Convert indices to feature names
top_feature_names = [input_features[i] for i in top_features]

# Count occurrences of each feature
feature_counts = Counter(top_feature_names)

# Sort the feature counts by frequency (value) in descending order
sorted_features = sorted(feature_counts.items(), key=lambda x: x[1], reverse=True)
sorted_feature_names, sorted_counts = zip(*sorted_features)

# Plot sorted histogram as horizontal bars
plt.figure(figsize=(10, 6))
bars = plt.barh(sorted_feature_names, sorted_counts, color='green')

# Add value labels on bars
for bar, count in zip(bars, sorted_counts):
    plt.text(
        bar.get_width() + 0.1,  # X-coordinate of text (just beyond the bar's length)
        bar.get_y() + bar.get_height() / 2,  # Y-coordinate of text (centered vertically on the bar)
        str(count),  # Text to display
        va='center', color='gray', fontsize=10
    )

plt.gca().invert_yaxis()  # Invert Y-axis to have the most important feature at the top
plt.title(f'Histogram of Top Feature Across {ats} Ensemble')
plt.xlabel('Frequency')
plt.ylabel('Features')
plt.savefig(rf"C:\Users\vwgei\Documents\PVOCAL\ensemble\regional_ensemble\{PASTEL_version}_{ats}\{ats}_re_feature_importances_composite.png", dpi=300, bbox_inches='tight')
plt.tight_layout()
plt.show()

In [ ]:
print(df['CH3Br'].mean())
print(df['CH3Br'].std())
print(df['CH3Br'].var())


In [ ]:
# Calculate mean and standard deviation
mean = df['CH3Br'].mean()
std = df['CH3Br'].std()

# Calculate range for 2 standard deviations
lower_bound = mean - 2 * std
upper_bound = mean + 2 * std

# Filter and count
within_range = df[(df['CH3Br'] >= lower_bound) & (df['CH3Br'] <= upper_bound)]
count_within_range = within_range.shape[0]

print(f"Number of samples within 2 standard deviations: {count_within_range}")

In [ ]:
non_nan_count = df['CH3Br'].notna().sum()
print(f"Number of non-NaN values: {non_nan_count}")

In [ ]:
average_concentration = df.groupby('domain_indicator')['CH3Br'].mean().reset_index()
print(average_concentration.sort_values(by="CH3Br"))

In [ ]:
from matplotlib.colors import ListedColormap

# Extract the colors from tab20b
tab20_colors = plt.cm.tab20b(np.linspace(0, 1, 20))

# Add a 21st color by interpolating (or define a custom color explicitly)
extra_color = np.array([[0.5, 0.5, 0.5, 1.0]])  # A neutral gray as an example
# extra_color = np.array([[0.0, 1.0, 1.0, 1.0]])  # Cyan: full green, full blue, no red
custom_colors = np.vstack([tab20_colors, extra_color])

# Create a custom ListedColormap
custom_tab20_21 = ListedColormap(custom_colors, name='custom_tab20_21')


# Example dictionary mapping numeric labels to custom string labels
label_mapping = {
    0: "ATom",
    1: "ACCLIP",
    2: "DC3",
    3: "DISCOVER-AQ_FRAPPE",
    4: "FIREX-AQ",
    5: "INTEX-A",
    6: "INTEX-B (C130)",
    7: "INTEX-B (DC8)",
    8: "KORUS-AQ",
    9: "PEM TROPICS A (DC8)",
    10: "PEM TROPICS A (P3)",
    11: "PEM TROPICS B (DC8)",
    12: "PEM TROPICS B (P3)",
    13: "PEM WEST A",
    14: "PEM WEST B",
    15: "SEAC4RS (DC8)",
    16: "TRACE A (DC8)",
    17: "TRACE A (P3)",
    18: "TRACE P (DC8)",
    19: "TRACE P (P3)",
    20: "WINTER",
    # Add more mappings as needed
}


# Drop any rows with missing CH4 data
mtest = df.dropna(subset="CH3Br")

# Ensure 'Datetime' is a datetime object
mtest['Datetime'] = pd.to_datetime(mtest['Datetime'])

# Extract the year and month to calculate yearly averages
mtest['Year'] = mtest['Datetime'].dt.year

# Group by 'Year' and calculate the mean of 'CH4' for each year
yearly_avg = mtest.groupby('Year')["CH3Br"].mean().reset_index()

# Step 1: Convert Yearly Averages to Datetime (Set date to the middle of the year)
yearly_avg['Datetime'] = pd.to_datetime(yearly_avg['Year'].astype(str) + '-06-01')

# Step 2: Plot data and yearly averages
plt.figure(figsize=(10, 6))

# Scatter plot of the data
sc = plt.scatter(df['Datetime'], df["CH3Br"], c=df['domain_indicator'].values,  # Use numeric values for consistent coloring
    cmap=custom_tab20_21, alpha=0.7, edgecolor='none')

# Step 3: Plot the yearly averages as datetime points
plt.scatter(yearly_avg['Datetime'], yearly_avg["CH3Br"], color='red', label='Yearly Average', zorder=5)

# Step 4: Annotate only the first and last yearly averages
first_point = yearly_avg.iloc[0]
last_point = yearly_avg.iloc[-1]

# Annotating the first point
plt.annotate(f"{first_point['CH3Br']:.0f}",  # Display CH4 value with two decimal places
             (first_point['Datetime'], first_point["CH3Br"]),  # Position the annotation at the point
             textcoords="offset points",  # Position relative to the point
             xytext=(0, 50),  # Slightly offset the label
             ha='center',  # Horizontal alignment to center
             fontsize=9,  # Font size of the label
             color='black')  # Color of the label

# Annotating the last point
plt.annotate(f"{last_point['CH3Br']:.0f}",  # Display CH4 value with two decimal places
             (last_point['Datetime'], last_point["CH3Br"]),  # Position the annotation at the point
             textcoords="offset points",  # Position relative to the point
             xytext=(0, 50),  # Slightly offset the label
             ha='center',  # Horizontal alignment to center
             fontsize=9,  # Font size of the label
             color='black')  # Color of the label

# Update the colorbar to match the new number of labels
cbar = plt.colorbar(sc, orientation='vertical', pad=0.05, aspect=30, shrink=0.7)
cbar.set_label("Campaign Name")
cbar.set_ticks(range(len(label_mapping)))
cbar.set_ticklabels([label_mapping[i] for i in range(len(label_mapping))])
cbar.ax.invert_yaxis()  # Flip the colorbar to align labels properly


# Step 5: Customize the plot
plt.title('CH3Br Mixing Ratios Over Input Dataset', fontsize=16)
plt.xlabel('Datetime', fontsize=14)
plt.ylabel('CH3Br (pptv)', fontsize=14)
plt.grid(True, linestyle='--', alpha=0.6)
plt.xticks(rotation=45, fontsize=10)
plt.legend(fontsize=12)
plt.tight_layout()

# Step 6: Show the plot
plt.show()

In [ ]:
from sklearn.metrics import r2_score
from sklearn.metrics import d2_absolute_error_score
from sklearn.metrics import mean_squared_error
# from sklearn.metrics import root_mean_squared_error
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import explained_variance_score
from sklearn.metrics import max_error

# Define a function that calcualte error metrics from predicted and actual values
def reg_model_metrics(actual,pred):
    MSE = mean_squared_error(actual,pred)
    RMSE = np.sqrt(MSE)
    actual_mean = np.mean(actual)
    RRMSE = 100*RMSE/actual_mean
    MAE = mean_absolute_error(actual, pred)
    R2 = r2_score(actual,pred)
    D2 = d2_absolute_error_score(actual, pred)
    MAXErr = max_error(actual, pred)
    EVS = explained_variance_score(actual, pred)
    return MSE, RMSE, RRMSE, MAE, R2, D2, MAXErr, EVS 

def scatter_plot(actual, pred, title, var_of_int, unit_label, comp_flag, save_path):
    # Color Matching for clarity
    if var_of_int == 'CH4':
        color = 'blue'
        cm = 'Blues'
    elif var_of_int == 'DMS':
        color = 'purple'
        cm = 'Purples'
    elif var_of_int == 'CO':
        color = 'cyan'
        cm = 'YlGnBu'
    elif var_of_int == 'O3':
        color = 'red'
        cm = 'YlOrRd'
    elif var_of_int == 'CH3Br':
        color = 'mediumseagreen'
        cm = 'Greens'
    elif var_of_int == 'Ethane':
        color = 'orange'
        cm = 'Oranges'
    else:
        color = 'gray'
        cm = 'Greys'
        
    if comp_flag:
        cm = "Greys"
    
    MSE, RMSE, RRMSE, MAE, R2, D2, MAXErr, EVS = reg_model_metrics(actual, pred)
        
    fig,ax = plt.subplots(figsize=(8, 6))
    ax.scatter(actual, pred, edgecolors=(0,0,0), c=color, cmap=cm)
    ax.plot([actual.min(), actual.max()], [actual.min(), actual.max()], 'r--', lw=2)
    text = r"R2 = %.2f" % (R2); text += "\n";
    text += r"D2 = %.2f" % (D2); text += "\n";
    text += r"MAE = %.2f" % (MAE); text += "\n";
    text += r"MSE = %.2f" % (MSE); text += "\n";
    text += r"RMSE = %.2f" % (RMSE);     
    plt.annotate(text, xy=(0.01, 0.9), xycoords='axes fraction',color='black', fontsize=10,bbox=dict(facecolor='none', edgecolor='none'))
    ax.set_xlabel(f'Measured {var_of_int} {unit_label}')
    ax.set_ylabel(f'Predicted {var_of_int} {unit_label}')
    # Set log scale for x and y axes
    ax.set_xscale('log')
    ax.set_yscale('log')
    # if var_of_int != 'DMS':
    # Create locators to specify the ticks on both axes
    ax.xaxis.set_major_locator(ticker.LogLocator(base=10.0, numticks=10))  # Controls major ticks
    ax.xaxis.set_minor_locator(ticker.LogLocator(base=10.0, subs="auto"))  # Controls minor ticks

    ax.yaxis.set_major_locator(ticker.LogLocator(base=10.0, numticks=10))
    ax.yaxis.set_minor_locator(ticker.LogLocator(base=10.0, subs="auto"))

    # Ensure minor gridlines are visible and major gridlines are configured
    ax.grid(visible=True, which="major", linestyle="--", linewidth=0.5)  # Major gridlines
    ax.grid(visible=True, which="minor", linestyle=":", linewidth=0.5)  # Minor gridlines
    ax.set_title(title)
    # plt.grid()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    # plt.show()
    plt.clf()
    plt.close()
    return None

In [ ]:
# Open the file in binary read mode and load the object
with open(rf"C:\Users\vwgei\Documents\PVOCAL\ensemble\global_models\{PASTEL_version}_Ethane\data\Ethane_global_y_train.pkl", 'rb') as file:
    ethane_train_y = pickle.load(file)
with open(rf"C:\Users\vwgei\Documents\PVOCAL\ensemble\global_models\{PASTEL_version}_Ethane\data\Ethane_global_y_test.pkl", 'rb') as file:
    ethane_test_y = pickle.load(file)

ethane_pred_train = np.load(fr"C:\Users\vwgei\Documents\PVOCAL\ensemble\PASTEL_combined\{PASTEL_version}_Ethane\Ethane_PASTEL_combined_pred_train.npy")
ethane_pred_test = np.load(rf"C:\Users\vwgei\Documents\PVOCAL\ensemble\PASTEL_combined\{PASTEL_version}_Ethane\Ethane_PASTEL_combined_pred_test.npy")

In [ ]:
save_path = rf"C:\Users\vwgei\Documents\PVOCAL\ensemble\PASTEL_combined\{PASTEL_version}_Ethane"

# Visulize predictions on training set
title = f'Training Results: Ethane [PASTEL - combined]'
scatter_plot(ethane_train_y, ethane_pred_train, title, 'Ethane', 'pptv', False, os.path.join(save_path, f"PASTEL_Ethane_train_linear.png"))
# Visulize predictions on testing set
title = f'Testing Results: Ethane [PASTEL - combined]'
scatter_plot(ethane_test_y, ethane_pred_test, title, 'Ethane', 'pptv', False, os.path.join(save_path, f"PASTEL_Ethane_test_linear.png"))

In [ ]:
ch4_residual_train = np.load(r"C:\Users\vwgei\Documents\PVOCAL\ensemble\PASTEL_combined\v0_1_5_CH4\CH4_PASTEL_combined_residuals_train.npy")
ch3br_residual_train = np.load(r"C:\Users\vwgei\Documents\PVOCAL\ensemble\PASTEL_combined\v0_1_5_CH3Br\CH3Br_PASTEL_combined_residuals_train.npy")

In [ ]:
# # Calculate Moran's I and p-value
I_dms, p_value_dms = morans_I(dms_train, dms_residual_train, threshold=110000, crs_epsg=3395, permutations=1000)
print(f"Observed Moran's I [DMS - train]: {round(I_dms, 4)}")
print(f"P-value [DMS - train]: {round(p_value_dms, 4)}")
I_dms, p_value_dms = morans_I(dms_test, dms_residual_test, threshold=110000, crs_epsg=3395, permutations=1000)
print(f"Observed Moran's I [DMS - test]: {round(I_dms, 4)}")
print(f"P-value [DMS- test]: {round(p_value_dms, 4)}")

dms_snr = np.var(dms_pred_test) / np.var(dms_residual_test)
print(f"DMS SNR: {round(dms_snr, 4)}")

I_ethane, p_value_ethane = morans_I(ethane_train, ethane_residual_train, threshold=110000, crs_epsg=3395, permutations=1000)
print(f"Observed Moran's I [Ethane - train]: {round(I_ethane, 4)}")
print(f"P-value [Ethane - train]: {round(p_value_ethane, 4)}")
I_ethane, p_value_ethane = morans_I(ethane_test, ethane_residual_test, threshold=110000, crs_epsg=3395, permutations=1000)
print(f"Observed Moran's I [Ethane - test]: {round(I_ethane, 4)}")
print(f"P-value [Ethane - test]: {round(p_value_ethane, 4)}")

ethane_snr = np.var(ethane_pred_test) / np.var(ethane_residual_test)
print(f"Ethane SNR: {round(ethane_snr, 4)}")

I_co, p_value_co = morans_I(co_train, co_residual_train, threshold=110000, crs_epsg=3395, permutations=1000)
print(f"Observed Moran's I [CO - train]: {round(I_co, 4)}")
print(f"P-value [CO - train]: {round(p_value_co, 4)}")
I_co, p_value_co = morans_I(co_test, co_residual_test, threshold=110000, crs_epsg=3395, permutations=1000)
print(f"Observed Moran's I [CO - test]: {round(I_co, 4)}")
print(f"P-value [CO - test]: {round(p_value_co, 4)}")

co_snr = np.var(co_pred_test) / np.var(co_residual_test)
print(f"CO SNR: {round(co_snr, 4)}")

I_o3, p_value_o3 = morans_I(o3_train, o3_residual_train, threshold=110000, crs_epsg=3395, permutations=1000)
print(f"Observed Moran's I [O3 - train]: {round(I_o3, 4)}")
print(f"P-value [O3 - train]: {round(p_value_o3, 4)}")
I_o3, p_value_o3 = morans_I(o3_test, o3_residual_test, threshold=110000, crs_epsg=3395, permutations=1000)
print(f"Observed Moran's I [O3 - test]: {round(I_o3, 4)}")
print(f"P-value [O3 - test]: {round(p_value_o3, 4)}")

o3_snr = np.var(o3_pred_test) / np.var(o3_residual_test)
print(f"O3 SNR: {round(o3_snr, 4)}")

I_ch4, p_value_ch4 = morans_I(ch4_train, ch4_residual_train, threshold=110000, crs_epsg=3395, permutations=1000)
print(f"Observed Moran's I [CH4 - train]: {round(I_ch4, 4)}")
print(f"P-value [CH4 - train]: {round(p_value_ch4, 4)}")
I_ch4, p_value_ch4 = morans_I(ch4_test, ch4_residual_test, threshold=110000, crs_epsg=3395, permutations=1000)
print(f"Observed Moran's I [CH4 - test]: {round(I_ch4, 4)}")
print(f"P-value [CH4 - test]: {round(p_value_ch4, 4)}")

ch4_snr = np.var(ch4_pred_test) / np.var(ch4_residual_test)
print(f"CH4 SNR: {round(ch4_snr, 4)}")

I_ch3br, p_value_ch3br = morans_I(ch3br_train, ch3br_residual_train, threshold=110000, crs_epsg=3395, permutations=1000)
print(f"Observed Moran's I [CH3Br - train]: {round(I_ch3br, 4)}")
print(f"P-value [CH3Br - train]: {round(p_value_ch3br, 4)}")
I_ch3br, p_value_ch3br = morans_I(ch3br_test, ch3br_residual_test, threshold=110000, crs_epsg=3395, permutations=1000)
print(f"Observed Moran's I [CH3Br - test]: {round(I_ch3br, 4)}")
print(f"P-value [CH3Br - test]: {round(p_value_ch3br, 4)}")

ch3br_snr = np.var(ch3br_pred_test) / np.var(ch3br_residual_test)
print(f"CH3Br SNR: {round(ch3br_snr, 4)}")


# OUTPUT v0_1_5:
# Observed Moran's I [DMS - train]: 0.0196
# P-value [DMS - train]: 0.001
# Observed Moran's I [DMS - test]: 0.0493
# P-value [DMS- test]: 0.001
# DMS SNR: 3.8768
# Observed Moran's I [Ethane - train]: -0.0001
# P-value [Ethane - train]: 0.9201
# Observed Moran's I [Ethane - test]: 0.006
# P-value [Ethane - test]: 0.049
# Ethane SNR: 1.5185
# Observed Moran's I [CO - train]: 0.0109
# P-value [CO - train]: 0.001
# Observed Moran's I [CO - test]: 0.0533
# P-value [CO - test]: 0.001
# CO SNR: 0.944
# Observed Moran's I [O3 - train]: 0.0058
# P-value [O3 - train]: 0.001
# Observed Moran's I [O3 - test]: 0.0179
# P-value [O3 - test]: 0.001
# O3 SNR: 10.0166
# Observed Moran's I [CH4 - train]: 0.001
# P-value [CH4 - train]: 0.2507
# Observed Moran's I [CH4 - test]: 0.0664
# P-value [CH4 - test]: 0.001
# CH4 SNR: 17.1322
# Observed Moran's I [CH3Br - train]: 0.0348
# P-value [CH3Br - train]: 0.001
# Observed Moran's I [CH3Br - test]: -0.0011
# P-value [CH3Br - test]: 0.6613
# CH3Br SNR: 0.8033

In [ ]:
# List of ATS to process and color mapping
ats_list = ["CH3Br", "CH4", "CO", "DMS", "Ethane", "O3",]  # Add all your ATS here
ats_color_mapping = {"DMS": 'purple', "Ethane": "gold", "CH4": 'blue', "O3": 'red', "CO": 'cyan', "CH3Br": 'green'}

# Input feature names
input_features = [
    'mean_distance_to_t1000_cities', 'min_distance_to_t1000_cities',
    'Latitude_0', 'Longitude_0', 'AltP_meters_0',
    'Specific_Humidity_0', 'Potential_Temperature_0', 'mean_lcl', 'bearings_from_origin_-24',
    'sum_moisture_flux', 'mean_mixing_depth', 'sum_solar_radiation', 'domain_indicator', 
    'time_difference_seconds_0', 'cos_julian_day', 'hour_cos_0', 
]

# Initialize a dictionary to track feature occurrences by ATS
feature_ats_mapping = defaultdict(list)

# Loop through each ATS and process the data
for ats in ats_list:
    if ats == "CH3Br":
        with open(rf"C:\Users\vwgei\Documents\PVOCAL\ensemble\{ats}_cluster_stats.pkl", 'rb') as file:
            var_cluster_stats = pickle.load(file)
    elif ats == "Ethane":
        with open(rf"C:\Users\vwgei\Documents\PVOCAL\ensemble\{ats}_cluster_stats.pkl", 'rb') as file:
            var_cluster_stats = pickle.load(file)
    else:        
        with open(rf"C:\Users\vwgei\Documents\PVOCAL\ensemble\{ats}_cluster_stats.pkl", 'rb') as file:
            var_cluster_stats = pickle.load(file)

    test_mi = [var_cluster_stats[i]['moran_I_test_RF'] for i in var_cluster_stats.keys()]
    print(f"{ats}: Average RF testing Morans I: {np.nanmean(test_mi)}")

    efid = [var_cluster_stats[i]['sorted_idx'] for i in var_cluster_stats.keys()]

    # Extract the top features (e.g., top 3 indices for simplicity)
    top_features = []
    for entry in efid:
        top_features.extend(entry[-1:])  # Adjust slice to [-3:] for top 3 indices if needed

    # Convert indices to feature names and associate with ATS
    top_feature_names = [input_features[i] for i in top_features]
    for feature in top_feature_names:
        feature_ats_mapping[feature].append(ats)

# Count occurrences of each feature grouped by ATS
aggregate_feature_counts = {feature: Counter(ats_list) for feature, ats_list in feature_ats_mapping.items()}

# Calculate total counts for each feature
total_feature_counts = {feature: sum(counts.values()) for feature, counts in aggregate_feature_counts.items()}

# Sort features by total count in descending order
sorted_features_by_total = sorted(total_feature_counts.items(), key=lambda x: x[1], reverse=True)
sorted_features = [item[0] for item in sorted_features_by_total]

# Flatten the counts and colors for plotting, sorted by total count
features = []
colors = []
counts = []
bar_width = 0.15  # Bar width for multi-bar plot (you can adjust this as needed)
index = np.arange(len(sorted_features))  # Positions for the bars

# Adjust the gap between feature groups
gap_between_groups = 1.5  # Increase this value to add more space between features
adjusted_index = np.arange(len(sorted_features)) * gap_between_groups  # Adjusted positions for groups

# Prepare data for grouped bars
for feature in sorted_features:  # Use the sorted features
    ats_counts = aggregate_feature_counts[feature]
    for i, ats in enumerate(ats_list):
        count = ats_counts[ats] if ats in ats_counts else 0
        features.append(feature)
        colors.append(ats_color_mapping[ats])
        counts.append(count)

# # Normalization factors
# normalization_factors = {
#     "CH3Br": 82, "CH4": 82, "CO": 82,
#     "Ethane": 82, "O3": 82,
#     "DMS": 50
# }

# # Prepare data for grouped bars with normalization
# for feature in sorted_features:
#     ats_counts = aggregate_feature_counts[feature]
#     for i, ats in enumerate(ats_list):
#         raw_count = ats_counts[ats] if ats in ats_counts else 0
#         normalized_count = raw_count / normalization_factors[ats]
#         features.append(feature)
#         colors.append(ats_color_mapping[ats])
#         counts.append(normalized_count)

# Create the grouped bar plot
plt.figure(figsize=(14, 8))

# Plot bars for each feature and ATS with offset
for i, (feature, color, count) in enumerate(zip(features, colors, counts)):
    # Offset each ATS bar for each feature
    feature_index = sorted_features.index(feature)
    bar_position = adjusted_index[feature_index] + (i % len(ats_list)) * bar_width - (len(ats_list) // 2) * bar_width
    bar = plt.bar(bar_position, count, color=color, width=bar_width, alpha=0.7, edgecolor='black')

    # # Add value labels on bars
    # plt.text(
    #     bar_position,  # X-coordinate of text
    #     count + 0.2,  # Y-coordinate of text, adding some space above the bar
    #     str(count),  # Text to display
    #     ha='center', va='bottom', fontsize=10, color='black'  # Adjust text position and style
    # )

# Customize plot labels and title
plt.xticks(adjusted_index, sorted_features, rotation=45, ha='right')  # Rotate and align feature names
plt.title('Histogram of the Most Important Feature Across Ensemble Members for Each ATS')
plt.xlabel('Input Features')
plt.ylabel('Frequency')

# Add some space between feature names
plt.subplots_adjust(bottom=0.2)

# Add legend
handles = [plt.Rectangle((0, 0), 1, 1, color=ats_color_mapping[ats]) for ats in ats_list]
plt.legend(handles, ats_list, title="ATS")

plt.grid(axis='y')

# Tight layout
plt.tight_layout()

# Save the plot
plt.savefig(rf"C:\Users\vwgei\Documents\PVOCAL\plots\standalone\aggregated_feature_importances.png", dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
for ats in ats_list:
    with open(rf"C:\Users\vwgei\Documents\PVOCAL\ensemble\global_models\v0_1_5_{ats}\data\{ats}_global_perm_importances.pkl", 'rb') as file:
        perm_importances = pickle.load(file)
    with open(rf"C:\Users\vwgei\Documents\PVOCAL\ensemble\global_models\v0_1_5_{ats}\data\{ats}_global_perm_importances_index.pkl", 'rb') as file:
        perm_importances_index = pickle.load(file)

    print(ats)
    print()
    for i in perm_importances_index:
        print(i)
        print(input_features[i])
        print(perm_importances['importances_mean'][i] / perm_importances['importances_mean'].max())

In [ ]:
# Prepare data for plotting
feature_importance_dict = {feature: [] for feature in input_features}

# Extract normalized importances for each ATS
for ats in ats_list:
    # Load the pickle files for perm importances and index
    with open(rf"C:\Users\vwgei\Documents\PVOCAL\ensemble\global_models\v0_1_5_{ats}\data\{ats}_global_perm_importances.pkl", 'rb') as file:
        perm_importances = pickle.load(file)
    with open(rf"C:\Users\vwgei\Documents\PVOCAL\ensemble\global_models\v0_1_5_{ats}\data\{ats}_global_perm_importances_index.pkl", 'rb') as file:
        perm_importances_index = pickle.load(file)

    # Normalize importances
    max_importance = perm_importances['importances_mean'].max()
    normalized_importances = perm_importances['importances_mean'] / max_importance

    # Add importances to the feature dictionary
    for i in perm_importances_index:
        feature_name = input_features[i]
        feature_importance_dict[feature_name].append(normalized_importances[i])

# Handle missing values with zeros if a feature is not present in some ATS
for feature in feature_importance_dict:
    while len(feature_importance_dict[feature]) < len(ats_list):
        feature_importance_dict[feature].append(0)

# Calculate total normalized importance for each feature
total_importance = {feature: sum(importances) for feature, importances in feature_importance_dict.items()}

# Sort features by total importance (greatest to least)
sorted_features = sorted(total_importance, key=total_importance.get, reverse=True)
sorted_importance_dict = {feature: feature_importance_dict[feature] for feature in sorted_features}

# Plot settings
bar_width = 0.12
num_features = len(sorted_features)
x = np.arange(num_features)

# Create the grouped bar plot
plt.figure(figsize=(14, 8))

# Plot bars for each ATS with offset
for i, ats in enumerate(ats_list):
    importances = [sorted_importance_dict[feature][i] for feature in sorted_features]
    bar_position = x + (i - len(ats_list) / 2) * bar_width
    plt.bar(bar_position, importances, width=bar_width, color=ats_color_mapping[ats], label=ats, alpha=0.8, edgecolor='black')

# Customize plot labels and title
plt.xticks(x, sorted_features, rotation=45, ha='right')
plt.title('Normalized Permutation Importances Across All Global Models (Sorted by Total Importance)')
plt.xlabel('Input Features')
plt.ylabel('Normalized Permutation (Feature) Importance')

# Add legend
plt.legend(title="ATS", loc='upper right')

# Add grid and layout adjustments
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.subplots_adjust(bottom=0.25)
plt.tight_layout()

# Save the plot
plt.savefig(rf"C:\Users\vwgei\Documents\PVOCAL\plots\standalone\global_normalized_feature_importances.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Extract the colors from tab20b
tab20_colors = plt.cm.tab20b(np.linspace(0, 1, 20))

# Add a 21st color by interpolating (or define a custom color explicitly)
extra_color = np.array([[0.5, 0.5, 0.5, 1.0]])  # A neutral gray as an example
# extra_color = np.array([[0.0, 1.0, 1.0, 1.0]])  # Cyan: full green, full blue, no red
custom_colors = np.vstack([tab20_colors, extra_color])

# Create a custom ListedColormap
custom_tab20_21 = ListedColormap(custom_colors, name='custom_tab20_21')

# Example dictionary mapping numeric labels to custom string labels
label_mapping = {
    0: "ATom",
    1: "ACCLIP",
    2: "DC3",
    3: "DISCOVER-AQ_FRAPPE",
    4: "FIREX-AQ",
    5: "INTEX-A",
    6: "INTEX-B (C130)",
    7: "INTEX-B (DC8)",
    8: "KORUS-AQ",
    9: "PEM TROPICS A (DC8)",
    10: "PEM TROPICS A (P3)",
    11: "PEM TROPICS B (DC8)",
    12: "PEM TROPICS B (P3)",
    13: "PEM WEST A",
    14: "PEM WEST B",
    15: "SEAC4RS (DC8)",
    16: "TRACE A (DC8)",
    17: "TRACE A (P3)",
    18: "TRACE P (DC8)",
    19: "TRACE P (P3)",
    20: "WINTER",
    # Add more mappings as needed
}

In [ ]:
alt = df["ALT"]

fig = plt.figure(figsize=(12,8))
sc = plt.scatter(df.index, alt, marker=".", c=df['domain_indicator'], s=1, cmap=custom_tab20_21)
plt.tick_params(
    axis='x',          # changes apply to the x-axis
    which='both',      # both major and minor ticks are affected
    bottom=True,       # ticks along the bottom edge are off
    top=False,         # ticks along the top edge are off
    labelbottom=True) # labels along the bottom edge are off
plt.ylabel("Altitude (m)")
plt.xlabel("Sample Index")
# Update the colorbar to match the new number of labels
cbar = plt.colorbar(sc, orientation='vertical', pad=0.05, aspect=30, shrink=0.7)
cbar.set_label("Campaign Name")
cbar.set_ticks(range(len(label_mapping)))
cbar.set_ticklabels([label_mapping[i] for i in range(len(label_mapping))])
# cbar.ax.invert_yaxis()  # Flip the colorbar to align labels properly

plt.show()


# fig = plt.figure(figsize=(12,8))
# sc = plt.scatter(df.index, alt, marker="o", c=df['O3'], s=5, cmap='RdBu_r')
# plt.tick_params(
#     axis='x',          # changes apply to the x-axis
#     which='both',      # both major and minor ticks are affected
#     bottom=False,      # ticks along the bottom edge are off
#     top=False,         # ticks along the top edge are off
#     labelbottom=False) # labels along the bottom edge are off
# cbar = plt.colorbar()
# cbar.set_label("Ozone (ppb)")
# plt.ylabel("Altitude (m)")
# plt.show()

# fig = plt.figure(figsize=(12,8))
# sc = plt.scatter(df.index, alt, marker="o", c=df['Specific_Humidity_0'], s=5, cmap='plasma')
# plt.tick_params(
#     axis='x',          # changes apply to the x-axis
#     which='both',      # both major and minor ticks are affected
#     bottom=False,      # ticks along the bottom edge are off
#     top=False,         # ticks along the top edge are off
#     labelbottom=False) # labels along the bottom edge are off
# cbar = plt.colorbar()
# cbar.set_label("Specific Humidity (g/kg)")
# plt.ylabel("Altitude (m)")
# plt.show()

In [ ]:
df['Latitude_Bin'] = df['Latitude_0'].apply(lambda x: round(x * 2) / 2)
df['Longitude_Bin'] = df['Longitude_0'].apply(lambda x: round(x * 2) / 2)
# Use select_dtypes to include only numeric columns

In [ ]:
# Use groupby and apply only on numeric columns, then reset the index properly
df_avg_1deg = df.groupby('Latitude_Bin').mean(numeric_only=True).reset_index()
df_avg_1deg_lon = df.groupby('Longitude_Bin').mean(numeric_only=True).reset_index()

In [ ]:
df_avg_1deg['bearings_from_origin_-24'].describe()

In [ ]:
# Sort and reset index based on datetime
sorted_df = df.sort_values(by='Latitude_0').reset_index(drop=True)

# Function to smooth using rolling average (no normalization)
def smooth(df, column_name, window_size=5):
    df[column_name + '_smoothed'] = df[column_name].rolling(window=window_size, center=True).mean()
    return column_name + '_smoothed'

# Input features
gc_features = [
    'bearings_from_origin_-24',
    'Distance_ptp_-24',
    'Solar_Radiation_0',
    'Potential_Temperature_0',
    'Specific_Humidity_0',
    'Temperature_C_0',
    'AltP_meters_0'
]

xlabels = [
    'radians\n(relative wind direction proxy)',
    'meters\n(relative wind speed proxy)',
    'W/m^2',
    'Kelvin',
    'g/kg',
    'Celcius',
    'meters'
]

random.seed(43)
# Generate random colors for the number of features
random_colors = ['#%06X' % random.randint(0, 0xFFFFFF) for _ in range(len(gc_features))]

win_size = 100
lat_averaging = win_size * 0.5

# Initialize the plot with subplots
fig, ax = plt.subplots(1, len(gc_features), figsize=(18, 4), sharey=True)

# Loop through each feature and plot the smoothed version with a random color
for i, feature in enumerate(gc_features):
    smoothed_feature = smooth(sorted_df, feature, window_size=win_size)
    
    # Plot using a unique random color
    ax[i].plot(sorted_df[feature], sorted_df['LAT'], label=f'{feature}', color=random_colors[i],alpha=0.3)
    ax[i].plot(sorted_df[smoothed_feature], sorted_df['LAT'], label=f'{feature}', color=random_colors[i])
    
    # Set title and label for each subplot
    ax[i].set_title(f'{feature}')
    ax[i].set_xlabel(f"{xlabels[i]}")
    ax[i].grid(True, linestyle="--", color="gray", axis='y')

# Set the common y-axis label
ax[0].set_ylabel('Latitude (degrees)')

# Adjust layout for better spacing
plt.tight_layout()
plt.savefig(rf"C:\Users\vwgei\Documents\PVOCAL\plots\standalone\zonal_circulation_noisy.png", dpi=300, bbox_inches='tight')
# Show the plot
plt.show()

In [ ]:
# Input features
zonal_ats_features = [
    'CO',
    'CH4', 
    'Ethane',
    'O3',
    'DMS',
    'CH3Br',
    'AltP_meters_0',
]

xlabels = [
    'ppbv',
    'ppbv',
    'pptv',
    'ppbv',
    'pptv',
    'pptv',
    '(m)'
]

ats_colors =[
    'cyan',
    'blue',
    'gold',
    'red',
    'purple',
    'green',
    'hotpink'
]

win_size = 5
lat_averaging = win_size * 0.5

# Initialize the plot with subplots
fig, ax = plt.subplots(1, len(zonal_ats_features), figsize=(18, 4), sharey=True)

# Loop through each feature and plot the smoothed version with a random color
for i, feature in enumerate(zonal_ats_features):
    smoothed_feature = smooth(df_avg_1deg, feature, window_size=win_size)
    
    # Plot using a unique random color
    ax[i].plot(df_avg_1deg[feature], df_avg_1deg['LAT'], label=f'{feature}', color=ats_colors[i],alpha=0.3)
    ax[i].plot(df_avg_1deg[smoothed_feature], df_avg_1deg['LAT'], label=f'{feature}', color=ats_colors[i])
    
    # Set title and label for each subplot
    ax[i].set_title(f'{feature}')
    ax[i].set_xlabel(f"{xlabels[i]}")
    ax[i].grid(True, linestyle="--", color="gray", axis='y')

# Set the common y-axis label
ax[0].set_ylabel('Latitude (degrees)')

# Adjust layout for better spacing
plt.tight_layout()
plt.savefig(rf"C:\Users\vwgei\Documents\PVOCAL\plots\standalone\zonal_ats_res{lat_averaging}.png", dpi=300, bbox_inches='tight')
# Show the plot
plt.show()

In [ ]:
# Input features
zonal_ats_features = [
    'CO',
    'CH4', 
    'Ethane',
    'O3',
    'DMS',
    'CH3Br',
    'AltP_meters_0',
]

xlabels = [
    'ppbv',
    'ppbv',
    'pptv',
    'ppbv',
    'pptv',
    'pptv',
    '(m)'
]

ats_colors =[
    'cyan',
    'blue',
    'gold',
    'red',
    'purple',
    'green',
    'hotpink'
]

win_size = 25
lat_averaging = win_size * 0.5

# Initialize the plot with subplots
fig, ax = plt.subplots(1, len(zonal_ats_features), figsize=(18, 4), sharey=True)

# Loop through each feature and plot the smoothed version with a random color
for i, feature in enumerate(zonal_ats_features):
    smoothed_feature = smooth(sorted_df, feature, window_size=win_size)
    
    # Plot using a unique random color
    ax[i].plot(sorted_df[feature], sorted_df['LAT'], label=f'{feature}', color=ats_colors[i],alpha=0.3)
    ax[i].plot(sorted_df[smoothed_feature], sorted_df['LAT'], label=f'{feature}', color=ats_colors[i])
    
    # Set title and label for each subplot
    ax[i].set_title(f'{feature}')
    ax[i].set_xlabel(f"{xlabels[i]}")
    ax[i].grid(True, linestyle="--", color="gray", axis='y')

# Set the common y-axis label
ax[0].set_ylabel('Latitude (degrees)')

# Adjust layout for better spacing
plt.tight_layout()
plt.savefig(rf"C:\Users\vwgei\Documents\PVOCAL\plots\standalone\zonal_ats_noisy.png", dpi=300, bbox_inches='tight')
# Show the plot
plt.show()

In [ ]:
# Sort and reset index based on datetime
sorted_df = df.sort_values(by='Datetime').reset_index(drop=True)

# Function to smooth using rolling average (no normalization)
def smooth(df, column_name, window_size=5):
    df[column_name + '_smoothed'] = df[column_name].rolling(window=window_size, center=True).mean()
    return column_name + '_smoothed'

# Input features
gc_features = [
    'bearings_from_origin_-24',
    'Distance_ptp_-24',
    'Solar_Radiation_0',
    'Potential_Temperature_0',
    'Specific_Humidity_0',
    'Temperature_C_0',
    'AltP_meters_0'
]

xlabels = [
    'radians\n(relative wind direction proxy)',
    'meters\n(relative wind speed proxy)',
    'W/m^2',
    'Kelvin',
    'g/kg',
    'Celcius',
    'meters'
]

random.seed(43)
# Generate random colors for the number of features
random_colors = ['#%06X' % random.randint(0, 0xFFFFFF) for _ in range(len(gc_features))]

# Rolling window size
win_size = 1000  # Adjust as needed for the smoothing window

# Initialize the plot with subplots
fig, ax = plt.subplots(1, len(gc_features), figsize=(25, 4), sharey=False)

# Loop through each feature and plot the raw and smoothed data
for i, feature in enumerate(gc_features):
    smoothed_feature = smooth(sorted_df, feature, window_size=win_size)  # Apply rolling average
    
    # Plot raw data
    ax[i].plot(sorted_df.index, sorted_df[feature], label=f'{feature} (raw)', color=random_colors[i], alpha=0.3)
    
    # Plot smoothed data
    ax[i].plot(sorted_df.index, sorted_df[smoothed_feature], label=f'{feature} (smoothed)', color=random_colors[i])
    
    # Set title and label for each subplot
    ax[i].set_title(f'{feature}')
    ax[i].set_xlabel('Increasing Time 1991 -> 2022 (sorted index)')
    ax[i].set_ylabel(f'{xlabels[i]}')
    # ax[i].legend()
    ax[i].grid(True, linestyle="--", color="gray", axis='y')

# Adjust layout for better spacing
plt.tight_layout()
plt.savefig(rf"C:\Users\vwgei\Documents\PVOCAL\plots\standalone\global_time_series.png", dpi=300, bbox_inches='tight')
# Show the plot
plt.show()

In [ ]:
# Function to smooth using rolling average (no normalization)
def smooth(df, column_name, window_size=5):
    df[column_name + '_smoothed'] = df[column_name].rolling(window=window_size, center=True).mean()
    return column_name + '_smoothed'

# New input features for the zonal atmospheric time series
zonal_ats_features = [
    'CO',
    'CH4', 
    'Ethane',
    'O3',
    'DMS',
    'CH3Br',
    'AltP_meters_0',
]

xlabels = [
    'ppbv',
    'ppbv',
    'pptv',
    'ppbv',
    'pptv',
    'pptv',
    '(m)'
]

ats_colors = [
    'cyan',
    'blue',
    'gold',
    'red',
    'purple',
    'green',
    'hotpink'
]

win_size = 25  # Set the window size for smoothing

# Initialize the plot with subplots
fig, ax = plt.subplots(1, len(zonal_ats_features), figsize=(25, 4), sharey=False)

# Loop through each feature and plot the raw and smoothed data
for i, feature in enumerate(zonal_ats_features):
    smoothed_feature = smooth(sorted_df, feature, window_size=win_size)  # Apply rolling average
    
    # Plot raw data
    ax[i].plot(sorted_df.index, sorted_df[feature], label=f'{feature} (raw)', color=ats_colors[i], alpha=0.3)
    
    # Plot smoothed data
    ax[i].plot(sorted_df.index, sorted_df[smoothed_feature], label=f'{feature} (smoothed)', color=ats_colors[i])
    
    # Set title and label for each subplot
    ax[i].set_title(f'{feature}')
    ax[i].set_xlabel('Increasing Time 1991 -> 2022 (sorted index)')
    ax[i].set_ylabel(f'{xlabels[i]}')
    # ax[i].legend()
    ax[i].grid(True, linestyle="--", color="gray", axis='y')

# Adjust layout for better spacing
plt.tight_layout()
plt.savefig(rf"C:\Users\vwgei\Documents\PVOCAL\plots\standalone\ats_time_series.png", dpi=300, bbox_inches='tight')
# Show the plot
plt.show()

In [ ]:
# Step 2: Plot data and yearly averages
plt.figure(figsize=(10, 6))

# Scatter plot of the data
sc = plt.scatter(df['time_difference_seconds_0'], df["Latitude_0"], c=df['domain_indicator'].values,  # Use numeric values for consistent coloring
    cmap=custom_tab20_21, edgecolor='none', marker='.')

# Update the colorbar to match the new number of labels
cbar = plt.colorbar(sc, orientation='vertical', pad=0.05, aspect=30, shrink=0.7)
cbar.set_label("Campaign Name")
cbar.set_ticks(range(len(label_mapping)))
cbar.set_ticklabels([label_mapping[i] for i in range(len(label_mapping))])
cbar.ax.invert_yaxis()  # Flip the colorbar to align labels properly


# Step 5: Customize the plot
plt.title('Latitude Over Input Dataset')
plt.xlabel('Seconds elapsed since 1900')
plt.ylabel('Latitude (degrees)')
plt.grid(True, linestyle='--', alpha=0.6)
plt.xticks(rotation=45, fontsize=10)
# plt.legend(fontsize=12)
plt.tight_layout()
plt.savefig(r"C:\Users\vwgei\Documents\PVOCAL\plots\standalone\time_diff_gaps.png", dpi=300, bbox_inches='tight')
# Step 6: Show the plot
plt.show()

In [ ]:
# TO DO - Address comment 6 on revisions with below code.

# also - add in x_train, x_test "iindex" for cluster stats. DONE (dont need x_test I dont think - if we really do we now can calculate them....)

# We just made plots explaining the global model ^^ evidence of general circulation, now make plots for regional model.
#  work in koppen clusters?

# PLOT: accuracy with koppen clusters? work in trajectories again? airmass origin?

# Work in old spatial clusters just so we have them and can present more senarios?


In [ ]:
# List of ATSs to process
atss = ['DMS', 'Ethane',  'CO', 'CH4', 'O3','CH3Br', ]
# Define colors for each ATS
colors = ['purple', 'orange', 'cyan', 'blue', 'red', 'mediumseagreen']

# Base directories
base_dir_base = r"C:\Users\vwgei\Documents\PVOCAL\ensemble\regional_ensemble\v0_1_5_"
cluster_stats_base = r"C:\Users\vwgei\Documents\PVOCAL\ensemble"

# Create a plot for all ATS
plt.figure(figsize=(10, 6))

# Loop through each ATS
for ats, color in zip(atss, colors):
    print(f"Processing {ats}...")

    # Update base_dir and cluster_stats_path dynamically for each ATS
    base_dir = os.path.join(base_dir_base + ats, "data")
    cluster_stats_path = os.path.join(cluster_stats_base, f"{ats}_cluster_stats.pkl")

    # Load cluster stats
    with open(cluster_stats_path, 'rb') as file:
        cluster_stats = pickle.load(file)

    # Initialize empty lists
    all_iindex = []
    best_model = []
    n = []
    best_model_single = []

    # Loop through clusters and process data
    for cluster in cluster_stats.keys():
        cluster_path = os.path.join(base_dir, str(cluster))
        cluster_iindex_path = os.path.join(cluster_path, "cluster_iindex_train.pkl")
        
        # Load cluster_iindex
        with open(cluster_iindex_path, 'rb') as file:
            cluster_iindex = pickle.load(file)
        
        # Store values and best model indicator
        all_iindex.append(cluster_iindex.values)
        
        # 1 for RF, 0 for LR
        best_model_value = 1 if cluster_stats[cluster]['best_model'] == 'RF' else 0
        best_model.append(np.full_like(cluster_iindex, best_model_value))

        # Add sample size and single best model indicator
        n.append(cluster_stats[cluster]['n'])
        best_model_single.append(best_model_value)

    # Flatten arrays
    n_flat = np.array(n)
    best_model_flat = np.array(best_model_single)

    # Filter n values where best_model == 0 (LR)
    n_lr = n_flat[np.array(best_model_single) == 0]
    
    # Get the maximum value of n where best_model == LR
    max_n_lr = np.max(n_lr) if len(n_lr) > 0 else None

    if max_n_lr is not None:
        print(f"Max n for LR in {ats}: {max_n_lr}")
    else:
        print(f"No LR models found for {ats}!")

    # Plot n vs. best_model_single for this ATS
    plt.scatter(n, best_model_single, color=color, label=f"{ats}", alpha=0.7)

    print(f"Finished processing {ats}!\n")

# Finalize the plot
plt.title("Comparison of Best Models Across Regional Ensemble Members")
plt.xlabel("Sample Size (n)")
plt.ylabel("Best Model Indicator (1=RF, 0=LR)")
plt.legend(title="ATS")
plt.grid(alpha=0.3)
plt.savefig(r"C:\Users\vwgei\Documents\PVOCAL\plots\standalone\best_model_stats.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# # Initialize MinMaxScaler
# scaler = MinMaxScaler()

# # Function to normalize and smooth using rolling average
# def normalize_and_smooth(df, column_name, window_size=5):
#     # Normalize using MinMaxScaler
#     scaler = MinMaxScaler()
#     df[column_name + '_norm'] = scaler.fit_transform(df[[column_name]])

#     # Apply rolling average for smoothing
#     df[column_name + '_smoothed'] = df[column_name + '_norm'].rolling(window=window_size, center=True).mean()

#     return column_name + '_smoothed'

# # Input features
# gc_features = [
#     'bearings_from_origin_-24',
#     'Distance_ptp_-24',
#     'Solar_Radiation_0',
#     'Potential_Temperature_0',
#     'Relative_Humidity_0',
#     'Temperature_C_0'
# ]

# # Initialize the plot
# fig, ax = plt.subplots(figsize=(8, 8))

# # Loop through each feature and plot both the normalized and smoothed versions
# for feature in gc_features:
#     smoothed_feature = normalize_and_smooth(df_avg_1deg, feature, window_size=10)
#     ax.plot(df_avg_1deg[smoothed_feature], df_avg_1deg['LAT'], label=f'{feature}')

# # Labeling the plot
# ax.set_ylabel('Latitude (degrees)')  # Set the y-axis label
# ax.set_title('Observed Zonal Circulation')  # Set the plot title

# # Add gridlines with dashed lines
# ax.grid(True, linestyle="--", color="gray", axis='y')

# # Place the legend outside the plot (right side)
# plt.legend(bbox_to_anchor=(0.0, -0.15), loc='center left', borderaxespad=0.)

# # Add annotation with a description
# plt.annotate('5° latitude averaging\nwith normalization', xy=(0.6, -0.15), xycoords='axes fraction', textcoords='axes fraction')

# # Show the plot
# plt.show()

# Initialize MinMaxScaler
scaler = MinMaxScaler()

# Input features
gc_features = [
    # 'CO',
    # 'CH4', 
    # 'Ethane',
    'O3',
    # 'DMS',
    # 'CH3Br',
    # 'CO2'
]

# Initialize the plot
fig, ax = plt.subplots(figsize=(8, 8))

# Loop through each feature and plot both the normalized and smoothed versions
for feature in gc_features:
    smoothed_feature = smooth(df_avg_1deg, feature, window_size=1)
    ax.plot(df_avg_1deg[smoothed_feature], df_avg_1deg['LAT'], label=f'{feature}')

# Labeling the plot
ax.set_ylabel('Latitude (degrees)')  # Set the y-axis label
ax.set_title('Zonal O3 Concentration')  # Set the plot title

# Add gridlines with dashed lines
ax.grid(True, linestyle="--", color="gray", axis='y')

# Place the legend outside the plot (right side)
plt.legend(bbox_to_anchor=(0.0, -0.05), loc='center left', borderaxespad=0.)

# Add annotation with a description
# plt.annotate('5° latitude averaging', xy=(0.6, -0.15), xycoords='axes fraction', textcoords='axes fraction')

# Show the plot
plt.show()

In [ ]:
# input_features = [
#     'mean_distance_to_t1000_cities', 'min_distance_to_t1000_cities',
#     'Latitude_0', 'Longitude_0', 'AltP_meters_0',
#     'Specific_Humidity_0', 'Potential_Temperature_0', 'mean_lcl', 'bearings_from_origin_-24',
#     'sum_moisture_flux', 'mean_mixing_depth', 'sum_solar_radiation', 'domain_indicator', 
#     'time_difference_seconds_0', 'cos_julian_day', 'hour_cos_0', 
# ]

# for feature in input_features:
#     fig = plt.figure(figsize=(12,8))
#     plt.scatter(df['LAT'], df['ALT'], c=df[feature])
#     cbar = plt.colorbar()
#     cbar.set_label(f"{feature}")
#     plt.xlabel("Latitude (degrees)")
#     plt.ylabel("Altitude (m)")
#     plt.savefig(rf"C:\Users\vwgei\Documents\PVOCAL\plots\altitude_plots\{feature}_latitude_altitude.png")
#     plt.clf()
#     plt.close()

fig = plt.figure(figsize=(12,8))
plt.scatter(df['LAT'], df['ALT'], c=df['O3'])
cbar = plt.colorbar()
cbar.set_label(f"Ozone (ppbv)")
plt.title("O3 Concentration by Altitude (Input Dataset)")
plt.xlabel("Latitude (degrees)")
plt.ylabel("Altitude (m)")
plt.savefig(rf"C:\Users\vwgei\Documents\PVOCAL\plots\altitude_plots\{feature}_latitude_altitude.png", dpi=300, bbox_inches='tight')
plt.show()
plt.clf()
plt.close()

In [ ]:
df['koppen_cluster'].describe()

In [ ]:
ats = 'DMS'

rf_model = load(rf"C:\Users\vwgei\Documents\PVOCAL\ensemble\global_models\v0_1_5_{ats}\models\{ats}_Global_RF.joblib")
with open(rf"C:\Users\vwgei\Documents\PVOCAL\ensemble\global_models\v0_1_5_{ats}\data\{ats}_global_X_train.pkl", 'rb') as file:
    X_train = pickle.load(file)

# Split X_train into chunks
n_chunks = 10
X_chunks = np.array_split(X_train, n_chunks)

# Create a single explainer outside the function to avoid re-creating it multiple times
explainer = shap.TreeExplainer(rf_model, feature_perturbation="tree_path_dependent", 
                                model_output="raw", approximate=True)

# Function to calculate SHAP for each chunk
def compute_shap_chunk(chunk):
    # Calculate SHAP values for this chunk
    return explainer.shap_values(chunk)

# Run SHAP computation in parallel
shap_values_list = Parallel(n_jobs=10)(delayed(compute_shap_chunk)(chunk) for chunk in X_chunks)

# Concatenate results (axis=0 because we're stacking samples)
shap_values = np.concatenate(shap_values_list, axis=0)

# Generate summary plot
shap.summary_plot(shap_values, X_train)

# Number of features in the dataset
n_features = len(X_train.columns)

# Create a figure with subplots in a grid layout
fig, axes = plt.subplots(n_features, n_features, figsize=(70, 50))

# Loop through each pair of features in X_train and generate a dependence plot
for i, feature_x in enumerate(X_train.columns):
    for j, feature_y in enumerate(X_train.columns):
        shap.dependence_plot(
            feature_x, 
            shap_values, 
            X_train,
            interaction_index=feature_y,  # Color by feature_y
            show=False,  # Don't display each plot immediately
            ax=axes[i, j]  # Place the plot in the corresponding subplot
        )


# Adjust layout
plt.tight_layout()
plt.savefig(r"C:\Users\vwgei\Documents\PVOCAL\plots\SHAP\shappair_dms.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
ats = 'Ethane'

rf_model = load(rf"C:\Users\vwgei\Documents\PVOCAL\ensemble\global_models\v0_1_5_{ats}\models\{ats}_Global_RF.joblib")
with open(rf"C:\Users\vwgei\Documents\PVOCAL\ensemble\global_models\v0_1_5_{ats}\data\{ats}_global_X_train.pkl", 'rb') as file:
    X_train = pickle.load(file)

# Split X_train into chunks
n_chunks = 10
X_chunks = np.array_split(X_train, n_chunks)

# Create a single explainer outside the function to avoid re-creating it multiple times
explainer = shap.TreeExplainer(rf_model, feature_perturbation="tree_path_dependent", 
                                model_output="raw", approximate=True)

# Function to calculate SHAP for each chunk
def compute_shap_chunk(chunk):
    # Calculate SHAP values for this chunk
    return explainer.shap_values(chunk)

# Run SHAP computation in parallel
shap_values_list = Parallel(n_jobs=10)(delayed(compute_shap_chunk)(chunk) for chunk in X_chunks)

# Concatenate results (axis=0 because we're stacking samples)
shap_values = np.concatenate(shap_values_list, axis=0)

with open(rf"C:\Users\vwgei\Documents\PVOCAL\ensemble\shap_values_train_{ats}.pkl", "wb") as f:
    pickle.dump(shap_values, f)

# Generate summary plot
shap.summary_plot(shap_values, X_train)

# Number of features in the dataset
n_features = len(X_train.columns)

# Create a figure with subplots in a grid layout
fig, axes = plt.subplots(n_features, n_features, figsize=(70, 50))

# Loop through each pair of features in X_train and generate a dependence plot
for i, feature_x in enumerate(X_train.columns):
    for j, feature_y in enumerate(X_train.columns):
        shap.dependence_plot(
            feature_x, 
            shap_values, 
            X_train,
            interaction_index=feature_y,  # Color by feature_y
            show=False,  # Don't display each plot immediately
            ax=axes[i, j]  # Place the plot in the corresponding subplot
        )


# Adjust layout
plt.tight_layout()
plt.savefig(rf"C:\Users\vwgei\Documents\PVOCAL\plots\SHAP\shappair_{ats}.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
ats = 'CO'

rf_model = load(rf"C:\Users\vwgei\Documents\PVOCAL\ensemble\global_models\v0_1_5_{ats}\models\{ats}_Global_RF.joblib")
with open(rf"C:\Users\vwgei\Documents\PVOCAL\ensemble\global_models\v0_1_5_{ats}\data\{ats}_global_X_train.pkl", 'rb') as file:
    X_train = pickle.load(file)

# Split X_train into chunks
n_chunks = 10
X_chunks = np.array_split(X_train, n_chunks)

# Create a single explainer outside the function to avoid re-creating it multiple times
explainer = shap.TreeExplainer(rf_model, feature_perturbation="tree_path_dependent", 
                                model_output="raw", approximate=True)

# Function to calculate SHAP for each chunk
def compute_shap_chunk(chunk):
    # Calculate SHAP values for this chunk
    return explainer.shap_values(chunk)

# Run SHAP computation in parallel
shap_values_list = Parallel(n_jobs=10)(delayed(compute_shap_chunk)(chunk) for chunk in X_chunks)

# Concatenate results (axis=0 because we're stacking samples)
shap_values = np.concatenate(shap_values_list, axis=0)

with open(rf"C:\Users\vwgei\Documents\PVOCAL\ensemble\shap_values_train_{ats}.pkl", "wb") as f:
    pickle.dump(shap_values, f)

# Generate summary plot
shap.summary_plot(shap_values, X_train)

# Number of features in the dataset
n_features = len(X_train.columns)

# Create a figure with subplots in a grid layout
fig, axes = plt.subplots(n_features, n_features, figsize=(70, 50))

# Loop through each pair of features in X_train and generate a dependence plot
for i, feature_x in enumerate(X_train.columns):
    for j, feature_y in enumerate(X_train.columns):
        shap.dependence_plot(
            feature_x, 
            shap_values, 
            X_train,
            interaction_index=feature_y,  # Color by feature_y
            show=False,  # Don't display each plot immediately
            ax=axes[i, j]  # Place the plot in the corresponding subplot
        )


# Adjust layout
plt.tight_layout()
plt.savefig(rf"C:\Users\vwgei\Documents\PVOCAL\plots\SHAP\shappair_{ats}.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
ats = 'CH4'

rf_model = load(rf"C:\Users\vwgei\Documents\PVOCAL\ensemble\global_models\v0_1_5_{ats}\models\{ats}_Global_RF.joblib")
with open(rf"C:\Users\vwgei\Documents\PVOCAL\ensemble\global_models\v0_1_5_{ats}\data\{ats}_global_X_train.pkl", 'rb') as file:
    X_train = pickle.load(file)

# Split X_train into chunks
n_chunks = 10
X_chunks = np.array_split(X_train, n_chunks)

# Create a single explainer outside the function to avoid re-creating it multiple times
explainer = shap.TreeExplainer(rf_model, feature_perturbation="tree_path_dependent", 
                                model_output="raw", approximate=True)

# Function to calculate SHAP for each chunk
def compute_shap_chunk(chunk):
    # Calculate SHAP values for this chunk
    return explainer.shap_values(chunk)

# Run SHAP computation in parallel
shap_values_list = Parallel(n_jobs=10)(delayed(compute_shap_chunk)(chunk) for chunk in X_chunks)

# Concatenate results (axis=0 because we're stacking samples)
shap_values = np.concatenate(shap_values_list, axis=0)

with open(rf"C:\Users\vwgei\Documents\PVOCAL\ensemble\shap_values_train_{ats}.pkl", "wb") as f:
    pickle.dump(shap_values, f)

# Generate summary plot
shap.summary_plot(shap_values, X_train)

# Number of features in the dataset
n_features = len(X_train.columns)

# Create a figure with subplots in a grid layout
fig, axes = plt.subplots(n_features, n_features, figsize=(70, 50))

# Loop through each pair of features in X_train and generate a dependence plot
for i, feature_x in enumerate(X_train.columns):
    for j, feature_y in enumerate(X_train.columns):
        shap.dependence_plot(
            feature_x, 
            shap_values, 
            X_train,
            interaction_index=feature_y,  # Color by feature_y
            show=False,  # Don't display each plot immediately
            ax=axes[i, j]  # Place the plot in the corresponding subplot
        )


# Adjust layout
plt.tight_layout()
plt.savefig(rf"C:\Users\vwgei\Documents\PVOCAL\plots\SHAP\shappair_{ats}.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
ats = 'O3'

rf_model = load(rf"C:\Users\vwgei\Documents\PVOCAL\ensemble\global_models\v0_1_5_{ats}\models\{ats}_Global_RF.joblib")
with open(rf"C:\Users\vwgei\Documents\PVOCAL\ensemble\global_models\v0_1_5_{ats}\data\{ats}_global_X_train.pkl", 'rb') as file:
    X_train = pickle.load(file)

# Split X_train into chunks
n_chunks = 10
X_chunks = np.array_split(X_train, n_chunks)

# Create a single explainer outside the function to avoid re-creating it multiple times
explainer = shap.TreeExplainer(rf_model, feature_perturbation="tree_path_dependent", 
                                model_output="raw", approximate=True)

# Function to calculate SHAP for each chunk
def compute_shap_chunk(chunk):
    # Calculate SHAP values for this chunk
    return explainer.shap_values(chunk)

# Run SHAP computation in parallel
shap_values_list = Parallel(n_jobs=10)(delayed(compute_shap_chunk)(chunk) for chunk in X_chunks)

# Concatenate results (axis=0 because we're stacking samples)
shap_values = np.concatenate(shap_values_list, axis=0)

with open(rf"C:\Users\vwgei\Documents\PVOCAL\ensemble\shap_values_train_{ats}.pkl", "wb") as f:
    pickle.dump(shap_values, f)

# Generate summary plot
shap.summary_plot(shap_values, X_train)

# Number of features in the dataset
n_features = len(X_train.columns)

# Create a figure with subplots in a grid layout
fig, axes = plt.subplots(n_features, n_features, figsize=(70, 50))

# Loop through each pair of features in X_train and generate a dependence plot
for i, feature_x in enumerate(X_train.columns):
    for j, feature_y in enumerate(X_train.columns):
        shap.dependence_plot(
            feature_x, 
            shap_values, 
            X_train,
            interaction_index=feature_y,  # Color by feature_y
            show=False,  # Don't display each plot immediately
            ax=axes[i, j]  # Place the plot in the corresponding subplot
        )


# Adjust layout
plt.tight_layout()
plt.savefig(rf"C:\Users\vwgei\Documents\PVOCAL\plots\SHAP\shappair_{ats}.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
ats = 'CH3Br'

rf_model = load(rf"C:\Users\vwgei\Documents\PVOCAL\ensemble\global_models\v0_1_5_{ats}\models\{ats}_Global_RF.joblib")
with open(rf"C:\Users\vwgei\Documents\PVOCAL\ensemble\global_models\v0_1_5_{ats}\data\{ats}_global_X_train.pkl", 'rb') as file:
    X_train = pickle.load(file)

# Split X_train into chunks
n_chunks = 10
X_chunks = np.array_split(X_train, n_chunks)

# Create a single explainer outside the function to avoid re-creating it multiple times
explainer = shap.TreeExplainer(rf_model, feature_perturbation="tree_path_dependent", 
                                model_output="raw", approximate=True)

# Function to calculate SHAP for each chunk
def compute_shap_chunk(chunk):
    # Calculate SHAP values for this chunk
    return explainer.shap_values(chunk)

# Run SHAP computation in parallel
shap_values_list = Parallel(n_jobs=10)(delayed(compute_shap_chunk)(chunk) for chunk in X_chunks)

# Concatenate results (axis=0 because we're stacking samples)
shap_values = np.concatenate(shap_values_list, axis=0)

with open(rf"C:\Users\vwgei\Documents\PVOCAL\ensemble\shap_values_train_{ats}.pkl", "wb") as f:
    pickle.dump(shap_values, f)

# Generate summary plot
shap.summary_plot(shap_values, X_train)

# Number of features in the dataset
n_features = len(X_train.columns)

# Create a figure with subplots in a grid layout
fig, axes = plt.subplots(n_features, n_features, figsize=(70, 50))

# Loop through each pair of features in X_train and generate a dependence plot
for i, feature_x in enumerate(X_train.columns):
    for j, feature_y in enumerate(X_train.columns):
        shap.dependence_plot(
            feature_x, 
            shap_values, 
            X_train,
            interaction_index=feature_y,  # Color by feature_y
            show=False,  # Don't display each plot immediately
            ax=axes[i, j]  # Place the plot in the corresponding subplot
        )


# Adjust layout
plt.tight_layout()
plt.savefig(rf"C:\Users\vwgei\Documents\PVOCAL\plots\SHAP\shappair_{ats}.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# # If we got this to look at all like the KOPPEN clustering that could be interesting...

# from sklearn.cluster import OPTICS
# # Fit the OPTICS model
# optics = OPTICS(min_samples=5, xi=0.03, min_cluster_size=0.002, n_jobs=10)
# optics.fit(df[input_features])

# # Add the OPTICS labels as a new column in the DataFrame
# df['optics_label'] = optics.labels_

# # Generate random colors for each cluster (excluding noise points labeled as -1)
# unique_labels = np.unique(df['optics_label'].values)
# # Exclude noise points (-1)
# unique_labels = unique_labels[unique_labels != -1]

# # Create a dictionary mapping each label to a random color
# label_colors = {label: random.choice(list(mcolors.CSS4_COLORS.values())) for label in unique_labels}

# # Create the plot
# fig, ax = plt.subplots(figsize=(12, 8), subplot_kw={'projection': ccrs.PlateCarree(central_longitude=240)})
# # ax.set_extent([-136, 120, -90, 90], crs=ccrs.PlateCarree(central_longitude=240))

# # Add basemap features
# ax.coastlines(resolution='50m')
# ax.add_feature(cfeature.BORDERS.with_scale('50m'), linestyle=':')
# ax.add_feature(cfeature.LAND.with_scale('50m'), facecolor='lightgray')
# ax.add_feature(cfeature.OCEAN.with_scale('50m'), facecolor='white')

# # Add gridlines
# gl = ax.gridlines(draw_labels=True, linestyle="--", color="gray")
# gl.top_labels = gl.right_labels = False  # optional: turn off top/right labels for cleaner look

# # Map each data point to a color based on its cluster label
# colors = [label_colors[label] if label != -1 else 'gray' for label in df['optics_label'].values]

# # Plot the data with the assigned random colors
# sc = ax.scatter(
#     df['Longitude_-24'].values, 
#     df['Latitude_-24'].values, 
#     c=colors,  # Color points according to the random color map for each label
#     s=20, 
#     transform=ccrs.PlateCarree(),
# )

# # Title and labels
# ax.set_title(f"OPTICS clustering results with Random Colors")

# # Show the plot
# plt.show()

In [ ]:
# #Code for temporarily generating the "iindex" values for cluster training and testing subsets.

# ats_label = 'DMS'
# version_str = "0_1_5"
# var_of_int = 'DMS'

# global_data_save_path = os.path.join("C:\\", "Users", "vwgei", "Documents", "PVOCAL","ensemble", "global_models", f"v{version_str}_{ats_label}", "data")

# # awakens = pd.read_csv(r"C:\Users\vwgei\Documents\PVOCAL\data\v0_1_5\v0_1_5_Awakens.csv")

# # Base directories
# base_dir_base = r"C:\Users\vwgei\Documents\PVOCAL\ensemble\regional_ensemble\v0_1_5_"
# cluster_stats_base = r"C:\Users\vwgei\Documents\PVOCAL\ensemble"

# # Update base_dir and cluster_stats_path dynamically for each ATS
# base_dir = os.path.join(base_dir_base + ats_label, "data")
# cluster_stats_path = os.path.join(cluster_stats_base, f"{ats_label}_cluster_stats.pkl")

# # Load cluster stats
# with open(cluster_stats_path, 'rb') as file:
#     cluster_stats = pickle.load(file)

# for cluster in cluster_stats.keys():
#     cluster_data_save_path = os.path.join("C:\\", "Users", "vwgei", "Documents", "PVOCAL","ensemble", "regional_ensemble", f"v{version_str}_{ats_label}", "data", f"{cluster}")
#     # Open the file in binary read mode and load the object
#     with open(os.path.join(global_data_save_path, f"{ats_label}_global_X_test.pkl"), 'rb') as file:
#         temp_X_test = pickle.load(file)

#     with open(os.path.join(global_data_save_path, f"{ats_label}_global_X_test_iindex.pkl"), 'rb') as file:
#         temp_X_test_iindex = pickle.load(file)

#     temp_X_test['iindex'] = temp_X_test_iindex

#     X_train_with_koppen = temp_X_test.merge(df[['iindex', 'koppen_spatial_relabel_reclass', ats_label]], on='iindex', how='left')

#     cluster_df = X_train_with_koppen[X_train_with_koppen['koppen_spatial_relabel_reclass'] == cluster]
#     print(f"Processing ensemble member: {cluster}")

#     data_subset = cluster_df.dropna(subset=var_of_int)
#     # inverse_subset = cluster_df[~cluster_df.index.isin(data_subset.index)]

#     data_subset = data_subset.replace([np.inf, -np.inf], np.nan)
#     # inverse_subset = inverse_subset.replace([np.inf, -np.inf], np.nan)

#     data_subset = data_subset.dropna(subset=input_features)
#     # inverse_subset = inverse_subset.dropna(subset=input_features)

#     cluster_iindex = df.iloc[data_subset.index]['iindex']
#     heck = os.path.join(cluster_data_save_path, "cluster_iindex_test.pkl")
#     print(heck)
#     with open(heck, "wb") as f:
#         pickle.dump(cluster_iindex, f)

In [ ]:
# # Calculate the mean and standard deviation of the 'CO' column
# mean_CO = df['CO'].mean()
# std_CO = df['CO'].std()

# # Define the cutoff for 3 standard deviations
# cutoff_lower = mean_CO - 2 * std_CO
# cutoff_upper = mean_CO + 2 * std_CO

# # Filter the DataFrame to keep only values within 3 standard deviations
# df_filtered = df[(df['CO'] >= cutoff_lower) & (df['CO'] <= cutoff_upper)]

# plt.scatter(df_filtered['time_index'], df_filtered['CO'])

In [ ]:
# # Create the scatter plot
# plt.figure(figsize=(10, 6))
# plt.scatter(datetimes, CO, color='orange', alpha=0.7, edgecolor='k')

# # Customize the plot
# plt.title('Scatter Plot of CO Mixing Ratios Over Time', fontsize=16)
# plt.xlabel('Datetime', fontsize=14)
# plt.ylabel('CO (ppbv)', fontsize=14)
# plt.grid(True, linestyle='--', alpha=0.6)
# plt.xticks(rotation=45, fontsize=10)
# plt.tight_layout()
# # plt.xscale('log')
# # plt.yscale('log')

# # Show the plot
# plt.show()

In [ ]:
import pandas as pd
# def count_top_cities_by_country(csv_path, top_n=1000):
#     cities = pd.read_csv(csv_path)
#     print(cities['population'].dtype)
#     print(cities['population'].isna().sum())
#     top_n_cities = cities.sort_values('population', ascending=False).head(top_n)
#     return top_n_cities['country'].value_counts()

# result = count_top_cities_by_country(r"C:\Users\vwgei\Documents\PVOCAL\data\worldcities.csv")
# print(result)

cities = pd.read_csv(r"C:\Users\vwgei\Documents\PVOCAL\data\worldcities.csv")

# Clean up
cities['population'] = pd.to_numeric(cities['population'], errors='coerce')
cities = cities.dropna(subset=['population'])
cities['country'] = cities['country'].str.strip()

# Optional: remove potential duplicates
cities = cities.drop_duplicates(subset=['city', 'country'])

top_1000 = cities.sort_values('population', ascending=False).head(1000)

print(top_1000['country'].value_counts().head(10))

In [ ]:
cities

In [ ]:
grouped = cities.groupby(['admin_name', 'country'], as_index=False)['population'].max()

top_1000_grouped = grouped.sort_values('population', ascending=False).head(1000)

print(top_1000_grouped['country'].value_counts().head(10))

In [ ]:
unique_cities = cities.drop_duplicates(subset=['city_ascii'])

top_1000 = unique_cities.sort_values('population', ascending=False).head(1000)

print(top_1000['country'].value_counts().head(10))